# Someshwari sand-mining paper: Figures 4, 5, 6, 7, 9, 10, 11

Final v11 revision. Run the cells in order from the top; every cell is
self-contained, with its data embedded, and writes a PNG and a PDF to the
working directory.

## What changed in this revision

**Printed text size, Jim comment 261.** AGU prints a two-column figure at a
maximum of 170 mm, or 6.693 in, and asks for figure labelling at 8 pt at that
final size. Text specified at F pt on a canvas W inches wide therefore reaches
the page at `F * 6.693 / W` pt. The previous build drew the seven figures at
canvas widths from 7.2 to 15.3 in while holding font sizes in a narrow 10 to
14 pt band, so printed text ranged from 4.8 pt on Figure 5 to 10.2 pt on
Figure 11. Each cell now carries a `FIG_SCALE_W, FIG_SCALE_H` pair that
rescales the drawing canvas while the authored font sizes are left alone.
Shrinking the canvas at fixed point size is the same as enlarging the type on
the printed page, and it preserves every layout decision in relative terms.
All seven now print their tick and axis labels between 8.69 and 8.79 pt.

Figure 9 also gains a taller aspect. As a 1 x 3 strip it printed 50 mm tall at
full width, too short to hold legible type at any size; it is now 77 mm.

**Font sizes that printed under the AGU floor** were lifted: the panel (b)
legend in Figure 4 and the panel (d) legend in Figure 10 were set one point
below the shared scale, Figure 6 had two 10 pt notes, and Figure 11's legend
and era tags were at 9.2 and 9.5 pt.

**Defects found in the visual pass**

- Figure 4: the right-hand axis label was longer than panel (a) and ran up into
  the legend above it. Now set on two lines.
- Figure 6: the panel (b) and panel (c) y-axis labels were long enough to meet
  each other in the left margin. Both are now on two lines.
- Figure 9: the panel (b) note read "Pre-min. minimum", now "Pre-mining
  minimum".
- Figure 11: the y-axis now reads "Annual mean water-table depth", so the
  reader can see why its trend values differ from the seasonal ones in
  Figure 7.

**Layout repairs forced by the larger type.** Figure 4's panel (b) legend moved
to two columns with more headroom; Figure 7's four panel (a) annotations were
repositioned into bands the weekly series leaves clear, since it reaches
9.6 m bgl in 2012 to 2015 and about 4.4 m before 2010; Figure 10's rotated bar
labels and panel (d) notes were spaced apart.

**A gap in the checking.** Figure 9 carries its own copy of the collision
checker rather than using the shared one, and that copy had no
label-against-label test. Its pixel test hides every tracked label before
rendering, so two labels sitting on each other leave no ink to find. That is
why the panel (b) note overprinted the "(b)" panel letter across successive
builds while the figure reported clean. The missing test is added below, and
it immediately surfaced a second overlap between the "2012" tag and the "(a)"
panel letter in panel (a).


## Shared style module

Fixes the on-page text sizes and derives build-time sizes from the figure
width, and provides the collision checker used by Figures 4, 7 and 10.


In [ ]:
# @title Shared figure style and validation helpers
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy import stats

"""Shared type scale and label-collision checking, used by every figure below.

Type scale
----------
AGU prints a two-column figure at a maximum of 170 mm, or 6.693 in, and asks
for figure labelling at 8 pt at that final size. A figure drawn at W inches is
therefore reduced by 6.693 / W, and a font specified at F pt appears on the
page at F * 6.693 / W. To make every figure read the same on the printed page
we fix the ON-PAGE sizes and derive the build-time sizes from the figure width.

    build_pt = on_page_pt * W / 6.693

Collision checking
------------------
Two independent tests run after the figure is drawn:

1. geometric - each tracked label's bounding box is compared against registered
   obstacles (data points, densified line segments, bar rectangles, the legend)
   and against the axes frame.
2. pixel - the figure is redrawn with every tracked label hidden and the pixels
   under each label box are inspected. Any ink underneath is a failure. This
   catches arrowheads, marker edges and anything not registered by hand.
"""
import io as _io

import numpy as np
import matplotlib as mpl
from matplotlib.transforms import Bbox
from PIL import Image

PAGE_W = 170.0 / 25.4          # AGU two-column maximum, 6.693 in

ON_PAGE = {
    'base':   8.5,    # axis labels and tick labels
    'panel':  9.5,    # panel letters
    'annot':  8.0,    # in-panel annotations, at the AGU floor
    'legend': 8.0,
    'small':  8.0,    # era tags, short numeric labels
}


# Every rcParam except the display backend, captured once so each apply()
# call can fully reset state before restyling. Without this, an `extra`
# override from one figure (e.g. hidden spines) silently persists into the
# next figure built in the same kernel session, since rcParams.update() only
# touches the keys it is given.
_RC_DEFAULTS = {k: v for k, v in mpl.rcParamsDefault.items() if k != 'backend'}


def apply(fig_width_in, extra=None):
    """Set rcParams for a figure of the given width and return the size table."""
    mpl.rcParams.update(_RC_DEFAULTS)
    k = fig_width_in / PAGE_W
    fs = {name: round(pt * k) for name, pt in ON_PAGE.items()}
    lw = round(0.8 * k, 2)
    mpl.rcParams.update({
        'font.family': 'DejaVu Sans',
        'font.size': fs['base'],
        'axes.labelsize': fs['base'],
        'xtick.labelsize': fs['base'],
        'ytick.labelsize': fs['base'],
        'axes.titlesize': fs['base'],
        'axes.linewidth': lw,
        'xtick.major.width': lw, 'ytick.major.width': lw,
        'xtick.minor.width': lw * 0.75, 'ytick.minor.width': lw * 0.75,
        'xtick.major.size': 3.5 * k, 'ytick.major.size': 3.5 * k,
        'xtick.minor.size': 2.0 * k, 'ytick.minor.size': 2.0 * k,
        'legend.fontsize': fs['legend'],
        'savefig.dpi': 300,
    })
    if extra:
        mpl.rcParams.update(extra)
    fs['k'] = k
    return fs


class LayoutCheck:
    """Registers labels and obstacles, then verifies nothing overlaps."""

    def __init__(self, fig, pad_px=3.0, ink_threshold=245):
        self.fig = fig
        self.pad = pad_px
        self.ink_threshold = ink_threshold
        self.labels = []      # (name, artist, axes, must_be_inside)
        self.points = {}      # axes -> list of (x, y)
        self.rects = {}       # axes -> list of (x0, y0, x1, y1)
        self.legends = []     # (name, legend)

    # ---- registration -------------------------------------------------
    def label(self, name, artist, ax, inside=True):
        self.labels.append((name, artist, ax, inside))
        return artist

    def legend(self, legend, name='legend', inside=True):
        self.legends.append((name, legend, inside))
        return legend

    def pts(self, ax, xs, ys):
        self.points.setdefault(ax, []).extend(
            zip(np.atleast_1d(xs).astype(float), np.atleast_1d(ys).astype(float)))

    def seg(self, ax, x0, x1, y0, y1, n=400):
        self.pts(ax, np.linspace(x0, x1, n), np.linspace(y0, y1, n))

    def path(self, ax, xs, ys, n=40):
        xs = np.asarray(xs, dtype=float); ys = np.asarray(ys, dtype=float)
        for i in range(len(xs) - 1):
            self.seg(ax, xs[i], xs[i + 1], ys[i], ys[i + 1], n=n)

    def rect(self, ax, x0, y0, x1, y1):
        self.rects.setdefault(ax, []).append((x0, y0, x1, y1))

    def bars(self, ax, xs, heights, width=0.8, base=0.0):
        for x, h in zip(xs, heights):
            self.rect(ax, x - width / 2, base, x + width / 2, h)

    # ---- checking -----------------------------------------------------
    def _box(self, artist, rend):
        bb = artist.get_window_extent(rend)
        return Bbox.from_extents(bb.x0 - self.pad, bb.y0 - self.pad,
                                 bb.x1 + self.pad, bb.y1 + self.pad)

    def run(self, verbose=True):
        fig = self.fig
        fig.canvas.draw()
        rend = fig.canvas.get_renderer()
        fails = []

        boxes = []
        for name, art, ax, inside in self.labels:
            boxes.append((name, self._box(art, rend), ax, inside))
        for name, leg, inside in self.legends:
            boxes.append((name, self._box(leg, rend), leg.axes, inside))

        # 1. geometric
        for name, bb, ax, inside in boxes:
            pts = self.points.get(ax, [])
            if pts:
                disp = ax.transData.transform(np.array(pts))
                hit = [(px, py) for px, py in disp
                       if bb.x0 <= px <= bb.x1 and bb.y0 <= py <= bb.y1]
                if hit:
                    inv = ax.transData.inverted().transform(np.array(hit))
                    fails.append(f"{name}: covers {len(hit)} plotted point(s), "
                                 f"e.g. data {np.round(inv[0], 2).tolist()}")
            for x0, y0, x1, y1 in self.rects.get(ax, []):
                (dx0, dy0), (dx1, dy1) = ax.transData.transform([(x0, y0), (x1, y1)])
                if bb.overlaps(Bbox.from_extents(min(dx0, dx1), min(dy0, dy1),
                                                 max(dx0, dx1), max(dy0, dy1))):
                    fails.append(f"{name}: covers a bar/box near x={x0:.4g}")
                    break
            if inside:
                ab = ax.get_window_extent(rend)
                if not (ab.x0 <= bb.x0 and bb.x1 <= ab.x1
                        and ab.y0 <= bb.y0 and bb.y1 <= ab.y1):
                    fails.append(f"{name}: extends outside the axes frame")

        # labels overlapping each other
        for i in range(len(boxes)):
            for j in range(i + 1, len(boxes)):
                if boxes[i][1].overlaps(boxes[j][1]):
                    fails.append(f"{boxes[i][0]} overlaps {boxes[j][0]}")

        # 2. pixel: is anything drawn beneath each label?
        vis = [(a, a.get_visible()) for _, a, _, _ in self.labels]
        vis += [(l, l.get_visible()) for _, l, _ in self.legends]
        for a, _ in vis:
            a.set_visible(False)
        buf = _io.BytesIO()
        fig.savefig(buf, format='png', dpi=fig.dpi, facecolor='white')
        for a, v in vis:
            a.set_visible(v)
        buf.seek(0)
        under = Image.open(buf).convert('RGB')
        W, H = under.size
        arr = np.asarray(under)
        ink = (arr < self.ink_threshold).any(axis=2)
        for name, bb, ax, inside in boxes:
            ix0, ix1 = max(0, int(bb.x0)), min(W, int(bb.x1) + 1)
            iy0, iy1 = max(0, int(H - bb.y1)), min(H, int(H - bb.y0) + 1)
            if ix1 <= ix0 or iy1 <= iy0:
                continue
            sub = ink[iy0:iy1, ix0:ix1]
            n = int(sub.sum())
            if n:
                ys_, xs_ = np.nonzero(sub)
                cols = arr[iy0:iy1, ix0:ix1][sub]
                darkest = cols[cols.min(axis=1).argmin()]
                px_x, px_y = ix0 + int(xs_[0]), H - (iy0 + int(ys_[0]))
                dx, dy = ax.transData.inverted().transform((px_x, px_y))
                fails.append(f"{name}: {n} inked px, darkest RGB {tuple(int(v) for v in darkest)}, "
                             f"first at data ({dx:.4g}, {dy:.4g})")

        if fails:
            print("\nLAYOUT FAILURES:")
            for f in fails:
                print("  -", f)
        elif verbose:
            print("layout ok: no label overlaps any point, line, bar, legend "
                  "or axes frame")
        return fails


## Figure 4: bar-area decline

Panel (b) legend moved to two columns with added headroom, and the
right-hand axis label set on two lines so it no longer runs into the
legend above panel (a).


In [ ]:
# @title Figure 4 - bar-area decline
"""Figure 4 v4 - bar-area decline.

Change from fig4_barArea_3panel: type scale only (Jim comment 261). The figure
is redrawn at the standard 11 in width so its on-page text matches the rest of
the set. Data, regressions, colours and panel content are unchanged.
"""
import gzip, base64, io, sys
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# ---------------------------------------------------------------- chat 31
# FIG_SCALE rescales the drawing canvas while the font sizes below are left as
# authored. AGU prints a two-column figure at 170 mm (6.693 in), so text set at
# F pt on a canvas W inches wide reaches the page at F * 6.693 / W pt. Scaling
# the canvas down is therefore the same as enlarging the type on the printed
# page, and it preserves every layout decision in relative terms.
FIG_SCALE_W, FIG_SCALE_H = 1.025, 1.025
# ---------------------------------------------------------------------------
_STYLE_W, _STYLE_H = 11.0, 8.6
FIG_W, FIG_H = _STYLE_W * FIG_SCALE_W, _STYLE_H * FIG_SCALE_H
FS = apply(_STYLE_W, {'axes.edgecolor': '#333333'})


def decode(b64):
    return pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(b64))))


LS_B64 = "H4sIAGyuWmoC/32XXW4cSQ6E3/csUiNJZpLJ0xgarDEwZjwDGAsM9vb7kdXd2odOGbK6VKoK8ScYwfzx8+P379/+/fGf72///f7x6+2fP9Xt28+3v3/9+P3HXx9/fvvt49e3P37q541/ePa69fH3j/r8l+Te70Peh77V5Vvepr7ZLWKFhZjqHDvzbdxWWgx+dPGMMefbvE3JadzeU4YPu2Ppu+QDyxpK0/ZeK4fbmA3la4ylusbU3Os1VHZYs6Cywmqo2LLGBmaPtRpKXAFZ5rpmfAUl+oCqBFN5QCsuUSflcZtTCIwndkb6yC+gdD+gpMLaAkaEDepDySqstSXE5+YPTKp1xKJY9sQqKA31HNPmnMM7w6kSw2ym+7I4Q9n7yAfU8Epxb6ueLXpIncAyIh1qy0Acoq+rlaMLH291ea/Wnmk5qb7BCLMKSwck8DEGN6naayjpanlBSfWwquXmc+qYSQDafAjTzPqme1kR5yWWNlalmFrUam5RqOKCze07AixXHp8rZqqFrjxiUS55YF1h8YqvzbuQ0zqsCTkC8toiUf0CSuIZ1i6sqTHFJUOmRU0PXWVsYm1PZ4heQ1nTdBWUVVSVobvw1xM6Ua1d3Iqxx1aDDlRLxzpi2TU9jaWV4nLaSAxMHcPtYHG5hgSE3zQ2jlDzfTzDIlfCIgj4GUa/clXhw8lVGcshEH8dsOZTa5LYxjU/jO9AVTLcRyi/vI2AIkwnn/BBjlh3rbmwmvSaAyotGQahpjfWnvTVd906xbU6rs5xPbFonVjGrrDoGkkyEVCVYdp8bkb3JZhXYD2M6QWmNUJVPUoflgv9qiRrCMFgjJbaOCQZnxMUhVWckJFDFqKwk0muHCE9YuYBYWOe+nhpc1Nid+0rLJ7RJYUnzEtNI22QHTN8JzO1/QhG/e0JVlgIej1EfVwG00iK02jt3DpnLj1AtQxew5jPFFMmarocviJ+Xa1qoKgOW6cuXoIaT6Rm/UDMlSDgBApTULCfl9Exo6/7CGV3QjSUeEe1ScQ2MUT5VveQAaDPFI+yfYGlnwmW0CMQO4rgNJCSFU9zLic1IbAtryQCyW1thvN12TytYRSPRAgEONfSj5tWP/d2BHraS0G9sPRdx4WVcLvoQH0xrynTA7upsGaUAKLxZbNfIe0nUgVVrJo7S52Rna4V3MyCIygZh6Aux5CCKsfgRaCcfwgVt4hpF9TQqAJuXKls8guofEDhxI21WZJmgFD+3FgExnJCmlD3gKV3jtZl132XzuNkubUWDxSiC1+m2C2s35wKr3eaPsHaF4uPCuf5D2W7YObCIDAEjKKdwZqoT7CLEPQMf3U6V8vKrUAXKxLrRYw4JNn+w0zX5aONJmidrVTEQFenuMLR0b2QaHz3C6iLpw3V04PMITblHGrSUbEkDURmsGTqPkel5WT/DwUjGLyxqft4EgJV5RZXakckvvYTqeZQasd1S6Q4/QqKOonW8kQfvgiKkb4H1X7RGQ6pnUFwx13rVQnzZq7VtTbpOBRrdlxaYJeRNR1Q+AkjmeLsQZSaQcYnF2/GIUX/LFZbj7XBkiTLm1WeEKklkHXcoQRCgV8esR7lumxstZqyUuZk6U4GvOvFUQFTZHmWNcehXtEK0dSKpyUiDKSI79QZg8xveKGzFvIak+52hGLhkidUK3P5KmOM2sH3VdaDWMF2LBG+xvQvsDQ+w2KObjwVgmVhYTxX9jqVFCliVAFP5WpHlC793V47MNSOjKi6ct7oJNnl2fHrHAP/8zXYdWLpJC9PxKcqstLUXcuaafncrRbN1thRbuJHsIc8X/5TSl9MrZUUfsaS5kRt0VkTxVLyOi5p/7HyDGn/se4jaxoNdEOeZ/s+wpWzOKG1Qr8OS6S5WrQn0cKq4rPb5sIRIRXTJ41FD9kHffPePmPBVf/E0irXxhcdh11LWLoKi+Mj8TCfCL4eoWYdfx5QHLwqLhQ+WRti4UVNVvqHKVGvDcKpXr3vFqu4LA/qhWRisCgCTIKi876+BZJodZ5xP0JpbYIPqNb5WTHRsFHHxtZ5fBLeLxvFsyOS3ZtYSNH7CCfOQB2kvvVOCUHY5DjGsMylvia9rPtxuC7vB6nWeKaHha9coxwD4eLLMUSYdUayyxQbafUWiCVjWvCAQ3AvSUhMHXyYJeqUrxVC/H6OqsvHOYqVDfljAFiZc7SekhgD7RzQdxyBMGp/AFlbDzYI+WvVVu0dEB7UGbHW8W0vd8D/AZ9aUJWpEQAA"
S2_B64 = "H4sIAGyuWmoC/32Y0Y4cuQ1F3/MtPQNRpEjpa4xZxFgYWWcBI8Aif59zVdtdGWCqHmw32l2nKfLykuofPz9+//7tnx//+f747/ePX4+//ujp334+/vz14/cf//7449tvH7++/etnP9/4i88eb338+UP//qM3y7fmbzYfevlY730+/L3MmvkyG93S89Heq83oIzLnnCvLHvEetsKX1wxr2Vysemv9zUysesz3NWBlps/RY6UN616CDW8jrM9aI2r517CpwJoLNoHN1aGNGe7Rbc1uncehTR/Who055vLucUebL1pu2ijz1ogs15ilg/If3pZb6lkbF7T11oyziragkTlgLVbsOEYtvqm9LwveUrCj8wV2x6oXS0kbpCz7gGVZHqHAZiPCCWz5zJk3MDsDC8GMcLwV+WqrqQCrNR9JEdJWLlJxwzoDU8YoVw6vbOjBUnEtX0WwNr0nhBtUP8Pa6TKfbqigT6S20zWJqCfH7t0o0xWqv7V8otpxxNATg+r7atZ3Iat6J7JYZWO160I+RbZpsWWBtqK1mKMhhXTRFplHXosUoui6o82DVuh/09pEED6DrgrzfVDeWuByceb1dc56e6W/N0ITygmgj8wR03P08bD3NnOYl/EfSOzimAdr5/9g2YZlbcyqRRfEhin7RaO7RfW8hP1dgQ2jeaD1JW3WmOgpOJKhly558SbHbj5vYNsyDpjHhgWfQv8Ilb6MTYvRqEun2Zu+8ormR18etO1m7msWj3HUWn2fE+fI6GWTA9e6YZ35t93keM+shs66j1A1BUvsg1Y3viStbmj9pLVNQwjqHZg5yJvOiTTUYPyFBHNc0oKSPmktNg1pTI6FUbT0KsXGYxR4NCmX89/RXgVtfdN4CLPxDASB4QPjlIZj88dR4td21m3b2Q7NBJsSLi0pDzTHVrNJtxLyzEpkRivMO1Y+WUc528L+8CHjzVahpHlvNQKl+KTnLrxRNHQ7XpEpruAT5Zyb3kxVs9cgT9TU6c24AVk7QLT5EokJ6atrJnFUDg0qZkqrLShoRN3BxguWueNqiL+jAUkqawfG6MXbgklFBvyS9hStaLMEo/hW+CCKraopGK7RTSMlifVKFxt2uMaGbV2YGicc7TOPYuxaDqJELsXExOvGHe0VWu15jjVrEaB800iXYOQMFyKdRG11HVq8lKHQYqsMc4a3iA+n2CpDb4uMYdnBab+G9e20G9aBDb1UIchZxtxSaAotkPBchdWxCFGJG5rbiybJmmysOSdc9KbKGep8zasRuNHFCBbrKQ6xSLsiW6oexdQyxDokGoqlK9ypEf1Ul7SnOx6045zB8Fyuz7a+9jGZzbyLWnxU3LHqyfIhFLnqDQFksmhsc6QQUzrr0p82zGuYnTCNJ5lfFHlLNfwBQymry5UajX/ltL6rufPvz8gKgaiR2M5a2FABUsfFxcgaqi2/gfUT5oJt1zLUjtsuVwfQE2zFFMH5nrbGHSxfMLDvWnVKaeusL31bI/XDKtURGC3deQl7uplgVA4YYzs0HekCXFssduy19B3Gc1cN4KehbVbplKuhT9dajeX7ThnpB6Ple7C3+B3sDCx3/guV05RVh9qV/+6pItNeuyeuYf2M7EgZchpNe13K1HbKlnxpGpaZve5S1sdnGHvx0NLDEFDjE5jhjGyl3IG0cl+inq20UVtkpY+Bow2JcJ+R5Gth1pdU3qHqE2qy5GgssgRpQAmlUc6uHYz4iYxvWHaG1cWiU5p2MXYWLEjZYgpg1pbai3JdouL/2gjn29ligmieMGgj9wIKl1shdx8UrLH1NS1O6YekDwv3QvfaFifGOJR6XmglYAlk47Ar64lTrfE8JP1CM/IZ9BWuEcesHDJwFMdt7mrExU6+PVmmwNLp6uBWSO9wpYHFE6MNYbgb5h0pP5NoIQKwxY6NLaqOjDn14mC2obi6C8vOsI4jYlRcrLmj8ODyfUYZD7bDgHLdrW9g+RkmTVbXBqCpqYYsLur4kEhLhEtY/N3dG6YeymBig8HAUiEJxn5W0wr7R7Z5kbOxrXpXcjylLzcI7jSYPt669BbKYrU6jKgzMW9g/YQpMj6PI3LZ4mTkR9OT0dF1T+dKqNvPLWy8YGIxcIsiGvPR+95VWDQYTRx5De4WEZcsnDqeLCWPgY2309OIo1hQlDIu9sq9aJ0XeQdbL5jvU3KUvn/GYC8bvk/Jnl37ZwxuOXUHszOynTIWOe3+uHOPY7tgqbLGqpGkjIXtFnZGJhbJISBbBMV4E4o20O8F7LYsj1fzaKN6fEKhJaqJRXNn6/OoJGpFvgxe1fa6ks8OF0upY/litdCSQlfPqXf0cxeDg/UOh7O4Q+ULNQ9VMA51a6gugRyq4AXXE64jNq5afMP6CYude9s/NHAnRQ1zp54BZ+xQKWv76oj/A4u5vUgHFAAA"
ls, s2 = decode(LS_B64), decode(S2_B64)

df = pd.concat([ls, s2], ignore_index=True)
df['image_date'] = pd.to_datetime(df['image_date'])
df = df.dropna(subset=['wl263_m', 'original_bar_km2']).reset_index(drop=True)
df['ydec'] = df['image_date'].dt.year + (df['image_date'].dt.dayofyear - 1) / 365.25
df['era'] = df['year'].apply(lambda y: 'pre' if y <= 2011 else ('early' if y <= 2016 else 'mech'))
pre, early, mech = df[df.era == 'pre'], df[df.era == 'early'], df[df.era == 'mech']

sl, ic, r, p, se = stats.linregress(pre['wl263_m'], pre['original_bar_km2'])
slm, icm, rm, _, _ = stats.linregress(mech['wl263_m'], mech['original_bar_km2'])
df['pred'] = sl * df['wl263_m'] + ic
df['resid'] = df['original_bar_km2'] - df['pred']
n = len(pre)
xm = pre['wl263_m'].mean()
ssx = np.sum((pre['wl263_m'] - xm) ** 2)
s_err = np.sqrt(np.sum((pre['original_bar_km2'] - (sl * pre['wl263_m'] + ic)) ** 2) / (n - 2))
tval = stats.t.ppf(0.975, n - 2)
print(f"pre-mining slope {sl:.4f} r {r:.4f} | mechanized slope {slm:.4f} r {rm:.4f}")
for e in ['pre', 'early', 'mech']:
    print(f"  {e} mean excess bar loss {df[df.era == e]['resid'].mean():.3f} km2")

C_PRE, C_EARLY, C_MECH = '#2c7fb8', '#f4a259', '#c0392b'
SH = {'pre': '#d6e6f2', 'early': '#fbe3c8', 'mech': '#f2d4d0'}
cmap = {'pre': C_PRE, 'early': C_EARLY, 'mech': C_MECH}
PRE_END, EARLY_END = 2011.5, 2016.5

fig = plt.figure(figsize=(FIG_W, FIG_H))
chk = LayoutCheck(fig)
chk.ink_threshold = 190
gs = fig.add_gridspec(3, 1, height_ratios=[1.0, 1.0, 0.85], hspace=0.30,
                      left=0.105, right=0.905, top=0.935, bottom=0.062)


def bands(ax, x0, x1):
    kw = dict(alpha=0.5, zorder=0, lw=0, antialiased=False)
    ax.axvspan(x0, PRE_END, color=SH['pre'], **kw)
    ax.axvspan(PRE_END, EARLY_END, color=SH['early'], **kw)
    ax.axvspan(EARLY_END, x1, color=SH['mech'], **kw)


def letter(ax, ch):
    return chk.label(f'{ch}:letter',
                     ax.text(0.988, 0.962, f'({ch})', transform=ax.transAxes,
                             fontsize=FS['panel'], fontweight='bold',
                             ha='right', va='top'), ax)


# ------------------------------------------------------------------ (a)
axa = fig.add_subplot(gs[0])
x0, x1 = 1987.5, 2025.8
bands(axa, x0, x1)
for era, label in [('pre', 'Pre-mining (1988\u20132011)'),
                   ('early', 'Early mining (2012\u20132016)'),
                   ('mech', 'Mechanized mining (2017\u20132025)')]:
    d = df[df['era'] == era]
    axa.scatter(d['ydec'], d['original_bar_km2'], s=42, color=cmap[era],
                edgecolor='white', linewidth=0.5, zorder=3, label=label)
axa.set_xlim(x0, x1); axa.set_ylim(2.15, 4.72)
axa.set_xlabel('Year')
chk.label('a:ylabel', axa.set_ylabel('Dry-season bar area (km\u00b2)'), axa, inside=False)
axa.grid(alpha=0.3, linewidth=0.6)

axa2 = axa.twinx()
axa2.scatter(df['ydec'], df['wl263_m'], s=14, color='#888888', alpha=0.7, zorder=2)
chk.label('a:ylabel2', axa2.set_ylabel('Water level at WLd\n(m PWD)', color='#666666'),
          axa2, inside=False)
axa2.tick_params(axis='y', colors='#666666')
axa2.set_ylim(6.6, 10.7)
axa.set_zorder(axa2.get_zorder() + 1)
axa.patch.set_visible(False)

# the grey water-level series covers the whole of (a), so the legend sits above it
lega = axa.legend(loc='lower left', bbox_to_anchor=(0.0, 1.055, 1.0, 0.12),
                  mode='expand', ncol=3, frameon=True, framealpha=0.95,
                  fontsize=FS['legend'], columnspacing=1.0, handletextpad=0.4,
                  borderaxespad=0.0)
chk.legend(lega, 'a:legend', inside=False)
letter(axa, 'a')
chk.pts(axa, df['ydec'].values, df['original_bar_km2'].values)
chk.pts(axa2, df['ydec'].values, df['wl263_m'].values)

# ------------------------------------------------------------------ (b)
axb = fig.add_subplot(gs[1])
xs = np.linspace(pre['wl263_m'].min() - 0.05, pre['wl263_m'].max() + 0.05, 100)
ys = sl * xs + ic
ci = tval * s_err * np.sqrt(1 / n + (xs - xm) ** 2 / ssx)
axb.fill_between(xs, ys - ci, ys + ci, color=C_PRE, alpha=0.15, zorder=1)
axb.plot(xs, ys, '-', color=C_PRE, linewidth=2.2, zorder=3,
         label=f'Pre-mining fit ({sl:+.2f}, r={r:.2f})')
xmf = np.linspace(mech['wl263_m'].min() - 0.05, mech['wl263_m'].max() + 0.05, 100)
axb.plot(xmf, slm * xmf + icm, '-', color=C_MECH, linewidth=2.2, zorder=3,
         label=f'Mechanized mining fit ({slm:+.2f}, r={rm:.2f})')
axb.scatter(pre['wl263_m'], pre['original_bar_km2'], s=20, facecolor='none',
            edgecolor=C_PRE, linewidth=0.8, alpha=0.7, zorder=5, label=f'Pre-mining (n={n})')
axb.scatter(early['wl263_m'], early['original_bar_km2'], s=55, color=C_EARLY,
            edgecolor='#7a4a10', linewidth=0.6, zorder=6, label=f'Early mining (n={len(early)})')
axb.scatter(mech['wl263_m'], mech['original_bar_km2'], s=55, color=C_MECH,
            edgecolor='#5a1a12', linewidth=0.6, zorder=6, label=f'Mechanized mining (n={len(mech)})')
axb.set_xlim(6.9, 10.6); axb.set_ylim(2.25, 5.95)
axb.set_xlabel('Water level at WLd (m PWD)')
chk.label('b:ylabel', axb.set_ylabel('Bar area (km\u00b2)'), axb, inside=False)
axb.grid(alpha=0.3, linewidth=0.6)
legb = axb.legend(loc='upper left', ncol=2, frameon=True, framealpha=0.95,
                  fontsize=FS['legend'], columnspacing=0.72, handletextpad=0.32,
                  borderaxespad=0.7)
chk.legend(legb, 'b:legend')
letter(axb, 'b')
for d in (pre, early, mech):
    chk.pts(axb, d['wl263_m'].values, d['original_bar_km2'].values)
chk.path(axb, xs, ys, n=4)
chk.path(axb, xmf, slm * xmf + icm, n=4)
chk.path(axb, xs, ys + ci, n=4)
chk.path(axb, xs, ys - ci, n=4)

# ------------------------------------------------------------------ (c)
axc = fig.add_subplot(gs[2])
ann = df.groupby('year').agg(resid=('resid', 'mean'), era=('era', 'first')).reset_index()
xd0, xd1 = 1987.3, 2025.8
bands(axc, xd0, xd1)
axc.bar(ann['year'], ann['resid'], width=0.75,
        color=[cmap[e] for e in ann['era']], edgecolor='white', linewidth=0.3, zorder=3)
axc.axhline(0, color='#333333', linewidth=0.9, zorder=2)
axc.axvspan(2011.62, 2013.38, color='#888888', alpha=0.18, zorder=1, lw=0, antialiased=False)
axc.set_xlim(xd0, xd1); axc.set_ylim(-3.10, 0.78)
axc.set_xlabel('Year')
chk.label('c:ylabel', axc.set_ylabel('Signed bar-area anomaly (km\u00b2)'), axc, inside=False)
axc.grid(alpha=0.3, linewidth=0.6, axis='y')
letter(axc, 'c')

TB = dict(boxstyle='round,pad=0.20', facecolor='white', edgecolor='none', alpha=0.0)
chk.label('c:gap', axc.text(2006.6, -1.05, 'No scenes\n(2012\u20132013)', ha='center',
                            va='center', fontsize=FS['annot'], color='#444444',
                            fontweight='bold', bbox=TB, zorder=6), axc)
axc.annotate('', xy=(2012.4, -0.30), xytext=(2010.0, -0.92),
             arrowprops=dict(arrowstyle='->', color='#444444', lw=1.0), zorder=6)
y2014 = ann.loc[ann['year'] == 2014, 'resid'].values
chk.label('c:signal', axc.text(2006.6, -2.15, 'Mining signal\nemerges (2014)', ha='center',
                               va='center', fontsize=FS['annot'], color='#333333',
                               fontweight='bold', bbox=TB, zorder=6), axc)
axc.annotate('', xy=(2014.0, y2014[0] - 0.08), xytext=(2010.4, -2.02),
             arrowprops=dict(arrowstyle='->', color='#333333', lw=1.0,
                             connectionstyle='arc3,rad=-0.15'), zorder=6)
chk.bars(axc, ann['year'].values, ann['resid'].values, width=0.75, base=0.0)
chk.seg(axc, xd0, xd1, 0, 0)

fig.align_ylabels([axa, axb, axc])

issues = chk.run()
if issues:
    print(f'\n{len(issues)} layout issue(s) above — figure not saved.')
else:
    _png = _io.BytesIO()
    fig.savefig(_png, format='png', dpi=300, bbox_inches='tight')
    with open('fig4_barArea_v4.png', 'wb') as _fh:
        _fh.write(_png.getvalue())
    fig.savefig('fig4_barArea_v4.pdf', bbox_inches='tight')
    print('saved fig4_barArea_v4.png and fig4_barArea_v4.pdf')
plt.show()
plt.close(fig)


## Figure 5: repeat cross-sections and thalweg elevation

Thalweg triangles on every survey, earlier surveys named by year in the
legends, consistent panel titles. Canvas reduced from 16 in to 8.4 in so
the type prints at the same size as the rest of the set.


In [ ]:
# @title Figure 5 - repeat cross-sections and thalweg elevation
# --- Embedded raw data (gzip + base64; newlines are ignored on decode) ---
import gzip, base64, io
import pandas as pd, numpy as np

_SUM = """H4sIANWQWmoC/4Vcu64Fqw3tI+VPtvYBg3n0t4mU2ySKUkZXUYo0SZH7/8pgM8NjlmdOedYCjPEDDLP/+vtvv//7v//5x59++fzy
2+//+vzy7//9/tt//vmvz9///Mc//OXXv/7tV/+JP0Q/5Fz5uI/P3+x8rTUlCw+1/SUCeKJGYBPJrjV1BRDy0Xf5ZtQ2h4/nb7GH
Lfnj/dcLoXhA8P5guEdGVoZrfwEJSCF/yjcIwVfUB9X8qad6fAWMwI2RVAyHGFElPYdBjOiEYasjJmEwhPgQwMsyJM6IkV+7z/Si
bnbUFvNBmexiY5AwcoIMbhbz1AcJIz0wOIjVsYUd6xniapH0wwfBO8MXVvimpQsOiouSUr3ju6dswM1RLvzmJwMRN6nSsgC8lLaw
pYq64h33PjUCqcoRIR8Kd/f/U0gfICnVZmtR5YkZ6IhL84a8G8FFiJ6hMwxC9I0QhRE8IHRXECOJhAhBHEJtCPWg/hBFSsqI4J/1
annDRMDOMBGwLwyCb4YO/m+4yCDcPGSFgIP4n+ZBPhoOssI3B7lgdRASB5lW7sJ3B9mAm4Nc+M1BLqQ4ySN+s4WBi4Mwb9Z24UdD
mCMGgRz0oEFos3bg/z2mm6J5dtATBuEwdJSdBiE3wt3QB6E66AkX4bDP5s/JnD6Rg/lvEKIQzFkSv3WQGwEsLYnwgaQpAc2HN9lC
cDBaDUJqhGTrJzqcOQchvKxyj1Yxbpl1EDTW2CYaa4vj31KEAHrgJmRE/8dBaiLgIDURcJAahDVIjf8bQWoQbkFqhUCQikroO9Zo
/D/f/3/svtIX9OOpAQkgRJJ2+Y4EQUATPoD4BS2O7eEBeADUI3YcvndHWjikbwRtvJN9WgCNvBfIIYhsqE3IfTOCot2KbShJhwlB
2R6rSIcFQVWgCiByoiiPIC8QUhSRqhdBQSCGUG4QkrCFQGMlKbYw/mVglcS+mTmyPha/RXoijjaUxeXAUMGJpSNzZkEIQUUhgFSZ
FHLNqCEnAiGidzaUm1fFAKAoDgfmG1X0wCbkB8Q/4YAoaIzw8P9HjCh3oG2g0xwKLkSDhENIaggYhSQWTPMcSJWIAwQIaVXABTTN
zInzAtq+4kh7FSAymQzaHGdIAykSydE0vVdtIijqVjAjiJsdOKhrMREiABUxLEJjVWmFlOpkqBwRlJqSElo9lwUCKiedcSgI0pyR
EZQkN0AbEsND607kV/+bEG7xBtmXRiJCUFII6aIIBBQYnCJA8rb5OiBkF0GlQHYRVAogX2hCHKEXQLEtI81BdPIBaRXBUFJTIehs
rWLSkgPwkFh0LIBU6Y8QwpINoSsWyddAdPayMawQkkRZCUFRTqawlZz1C2xVzFZ6vikRQZqvYSsRI6OxSMTIsMNittJ9NApA3Lco
AUHRlDCoGCgKhmIqSg9tBSG2MqKtjChSJCRFFCkSGktPhrBD1u0V0i6LGMhNmEUMRmKwvSZ6dEmoQ93l4VZqoWheScSoEBJHcajD
bDuKli2he+Ui+zXQYXJu28oNSOMJEj5pPCkoeWrsQmNlHSuDDrNGUGRRWSMoatRMI+L+kkBoWjnJjq0ipAiCRM/ehsjaFQl0bArO
VnRsy7xv9Vmpr7ZTVWxH5QzwtgmKZyndA4LuhXJuhCsezIQoKoheTtsJEJi1hzif52dCki1aKO04TmgI3zZxYT/xL4wiR4XsWx9X
FlwmGqWPJJWyUtFM2/6EHnXhhUG0FDEXRlPXcXLKNoPF68pa0l4Ype3zDkc3+4itiOD3YurCoPLWRzsLPcoRs8hRtVZREKP6F32w
C7qbqXN5c2EkYbCsLVz9RMJ4mEtqGjsO/GWpX8+MHMS7OZmzzVX9PyPjKSxgJZ4LRjOjkmxiao2WedUgDG+bV01yAVejyTiUJGtS
1gLmSomysA9m7F0RSnwYyHsZaKvCrZRY3gbyRSlrTWp17VDeZkQpv2juCBAsMdoOU55qK6IcCblVd+PcC8XrWhbGygUHczlxjaXA
K1cCUNhJCMEIthdBo3Hi+ZJkJmg0fhAyu7loNiNnlL376sUomuDva3XNsNnEEWWd2Udoseuxj+AluvWEQlBN8VOvLqAiOYI69cLI
jdHDDkFGjaCSvyjbNwZJuZwjkjSG1z44gqvkhSGSOrZnG7WU96Ax1lNlXG81F4YeE7ea+MxIegZ+WLlUxAu9beA9Dj/ZZ4/D1ZzL
GYztudQgO8AqnWSk0xoDuHxfGC37uafZajB2T/7sXVCKPV/vUpifGaxYDeAuZKF4spt7fh3dl9c5kH/VBEWlsD0Qpfqi8TOIPw6k
QZzSbbewU25xnn48XW8OQJzf8JvSBm7E+Z1w0+kgGHF+IuA4PwhGnB+EHuf3bdUgyDVsPC8i0RheT0thX4+J0YNG2jbmE0NLOMWb
DNLzX06mpMQ4aEwMLaf1NxCIcaYdW47Qaznrnd/CkHvX08ghQy83t4dJCyPn6ep1QSrBlDWtuVy+EmocRTT/MHDkBDPRxGiiof8X
B/PPzPAvquuVt1sgmxiJ9UbHNMek+5MHt5KiQNwfDixeceafbX8yMQrOLoNRXD8MmMZm5Z+JEd80VuWV0D0YLg7ca1ZmIPJOy1oP
OvNOaz0PKjnSkD7/eQgm/c0GQ4xfR5C7hmMq/iEYqQk9KeRMQ08ULbA9KP5KQ3aoMNPQnXJLQ/4nXg97QBZa4VvIu2AS/FYRGTjO
URt+s4sLNzLUwHGCunAjP1147vhmEhfes9MDQZMTAySZSO2VwbrWqoZWZOW+iTfnH4SAk9Ug9FxlKra9vWk9yMpEJEPFIWqo3khl
g6CZjNGqtqeCfWwGyxKyvs7ZTyODIMnglvAHXh3MYpNhvBEIvSCaCYxz3SAkB1PdIBwejtppGgN6Y4cj2CCoNXrT3lkTzy0BDgK/
LDuXlx6SuswDIUyPlub/F3zkmpxVN3E2odfw98A7CPxi9bnX8/e4fBF61iXTd8u5BJYC+7nR1k/pGjYdq7oXIa3EPwjRMrGqL+dv
Nbk5IuI91MToL+PYssJzPxDMhfKu6ONLFD77y00yV9Hry8yrJIIYrxL48jYL6pow18ETvemK9Npxe/O8MJIz1spLEPf3jebMwJuI
iSHvaBFgbC5uDFDK5OtZnuwtDGSpJPD5sEs2FACoGlUsZCmV8vkmSaqxoI3m/aVqw+c7O22EoKCbAYDoBRwQQXJ9HHe9s9ykEBiJ
WJYN6aeY4knGnp4lTVDQSYH+gl+eZSyIXBQAwYOGWaCHENZHGTOkl6gByaBRGSy6PBqhLzCgkM02/RCOZChRrnCQpcixryJErgUd
0EP0ck2KZhtlA4gh6gUPAOkVb0Ct+pkSQPoIGC2hHHcjtAj2vLzCnSFyNiQ3XGharM4GFoTV2Qiovd/HV9Bf8m6pUExAK1AQAuRr
KACQVg4Qwp+CTLztUQoylBRaC7B27ZGAAbQWjEZnqwmTIRazoZTk5g8sFkQ/vUAK06d/yFNkL5ZQf7IJS2idpaiR0MrI+4SElkbK
7AmuTSGrkeyg4EClWMLVYA7UocWoKV2PQ3k1zwVJCOkpLCPonqpWKKAOu2clAGneWUvdHSr9QQSAzporGsxreCI0mm+XpMdwsM8o
D0Pg3HxMNsbyRHK9+jqxco9fAwu3uHzpssc9JCdpjHVofj25IkgyKI9HlAumK+6RKJLbeLzLXLCk7TAWzKmHJMtQ4Xi5fdeaAoKK
XPHB0ap+eQdnV6UCyLDLqhfrGVpt0utyhJ0vWBBGt8w+oLRvSS6oVYICNoeot8EMpWTBCvQ79a4IpSz3HeSJcbcViD1EB0mh/I1o
Dqy2ArvMAsEec1xea61YsbG2UTIkaYl5epa+QmxJIod+XjcVF6bzhrpM6j/IiFKyo7NcAfN4LDdj2bE5XCbek84FRbd8WrBgmkmR
ofQMB43hTHEwfOtw0KBLskXpKdCjRSiVzT6r13bI7yrZffbsCV35zKxnu2Nd5heB7WDc6s5XkJvxpB8X6kMpRPCyqztOl8tHgiuj
6se+8jFvRYxSVQyeKwkzg2LVj0CXF30LQ29IAsW5YrIwyvSd6ArU+SZtgY7IWr4ICW7cjy3/l98aQIAWIdz6pGJhSDW1yMONqxw7
E2K/clzrsQujys2Jl4sEGn10Y9BnS2DBN3yrXM54wAYxCGIx/dXatNqD0O3hYYzLYpb7ioXRP+pc7+VmRr8g2ooyC6P/KMP65nFl
ZFB7WhhFvjx4mAvVN42SZvbtLcSqc1QlWxkM6oULw7/2oT8MQAyx9DJP/RZ4f4W1MuKbbUktv2y3ATND7wuQhJHf9BzlSr2mZ0br
Y305FdpFXrkegwDv2fCbDAMPeIKDoN6zXXfMBMt7boyb90yMhL6ZnhlkWNzEYOyBEwPWsBdGj/sPo1SpDNysatao++QXRutjvyFa
GO0eH/6fX+TrQf1Bk4EY1J4Xhlh93T4CnxlRvXu7DVoYuqI+LDXwhWFY/8qYKtxhuX0GVr/CN4OcbiihzU/XaYLrL32EO673DOD/
hifshJsjDAI7uDaDcF4+bIof17OEvWC9Ab49lpgJhpcMgjrJbd8wXRG3s2Z+WgApcPEeVCZCaITtinQhsDiZKAqtkXgQ8MKJwM+z
CPqtm61Jy5GmRwi6fdpccbus3q+7w3oZ/WhQ529NmCJEI7AOguGFC2FxwviTrrsacUID8ff/9535HdAkc/9/XRzt+rc3Ozpd0IZi
BlBPHQSg0+HAbLqrAaT3hxqdPzSEIDY0LS7pvwUgRX8JBDUqaa5uz0iVH4mBiGSmO9CTGiNEPBEA4oFo/OZ6Zd3WXYgls5xiPBQ6
qBMgcwhq/sCygu6CC4Cifq7KQA19T0fAUOJ2MT1DOlZACFvWFWsxbLwhm51QvorPR2/RQjwBpP+aEUDifUYn1PNUBlC52/4JnRf0
EKOHdn24grB8d9ALqzcPHbO2h+sntQi11e/lkShyOzt9sbxgKmaG7fT3TBxsV+VrE9ynfmU+7dAGFvRnnBAi3lcRQvPl1opIPvYQ
av5fcavcQgOG6ifjoWR35qHkgeWX5BAU5fl+CQhj/SiNIBb3AHFB99hxQcUtHzSvTtN/iw5hDzZ+/uAjgpLtNjGzaf5n3MkQ69Up
iEnoKQY0xZ74KVPhrv2ER9tW58s7ZjxJBpGXh+X6InkmtG/6juQTdVcfECOHo4/+jSzuo7ZRovxQ1NgXroxmr/vha2bQsflM5/4X
M7w/GI99+PrhW9FpYVB5kyPENpenPmpjJHkFWwgpPYTmZ1W0nkcXx4KWs7qGVm3G81f26bU6Briuan/pFAGh6LILISCCLLv1/8Mc
1mPYwhBziFuld2HIYjNE6rGEQXr3jHondxwkR+Bdofo5f5/NXUl1YRzCu37GrDlBxiFE39o7KDy1DDa8foVaYz2gO7huFMLb/EJ+
WVuK1MzrZqITI73Zh24c70Y8McRE9y8d74ybEdMPhbPIhYx4xpGQA1cj3n5WcCYUVNKZCasRb/8HRjwx5LHHPQ5MjCwq3I5oC6O8
zbA7wk3JM6OZQ1STqqgPauFgdogJakrun6/XGiCDD0aV6xJHSIVynPp69QgfIePIWd+QHhjtKNCPw7PbzQx+m2Z0kmT2iDUzml/k
rQq3MFIbZS9wzIx2OkJ+MTEMv9gZN7+YymDILSYY2cxcBTs6vylqevHclOB3q1yqZI8drD6z/hu4zCDIG6z9iX5cq2jPIyfsz0uZ
DdnA9KkG9qeJ0NzJ2ertLpPV8cE0mz+cH8fOCWIiUOuhj4Hw/Jm3ahNQoKetn5GA3DQRCPrZROCX6YuXMddtozYRsJMNgizi3cem
OqWD6WspQx4ybNWzmWC44EYA26t6FsKmn7Vb/p+/6f7vIPIAQNMs34Fq9NRO00eLCBB1DtCXz+1q2aPeSluLgnorfGwzEph624lP
1/8L0toEME7zifG7rDNADcgICSR7QITse786fpxz+vnCBeEPQ920it70NGRBWhskWXJWZ9kZTTJZksnzLUJIcVPJbjYmshY6iAJq
RUjEOlOg5n13JpWlzcYHkMeDsxkgDa4AiWYb9QA0TLn5xoloBuKKIPUPNJKPS6lzgdJS0lwgdR4koC9yjoNI+ozf4FoQ8Z5AUHsN
SkiI7iYQktgSIZItVWgMjlAK3l11IG1SyFLarijhRaRUWiPYKksggYgEH9if+EuAmqimxWpCqGhBtDaGbDZE0yjUa6A9N+gQ4f94
j+qAuWUAAA=="""
_SD  = """H4sIANWQWmoC/41bu44suQ3NDfhPugviQ6KU38SAnXgDh8aFscEm68D3/2GR6q6p6j7smgn7UCLFlyiy5rdfP3/98d8///23H7cf
P3/9fvvxx/9+/fzzP7/f/vX3v/7ln//47QfduFC5F76Xeis3km30Mf/MiKhzRtSL/9EoJjoAEX+Pqt9IN6Ygk05qgEjUiYYETalk
tSOq2CpEH416lSKASuVGvH3xawNxrM2pSpAx994JcWxt7RUca2tmSFt9KqJsPTRBpKoFKYJcXwUBrdzGJgc99obIRpCNOJeKMFtF
+qZ+g3ZoNpez+XKtZk0a0gsHFwod9zK4VVJkCql+5iuzWmhGg+kUoExFI4uVIJMrw9Ygqxc2U6PvmENt+gltLdyp9nkGgUy7W27j
dsF0qm2SjauTDvOYEY6j8tRPo5ys8pexjuqVO/Gdx+dQPhFlQfokoupULYmZJ9XHUH4SfQ7lnepjKD+pPofyk6q56tHvH4P3SbWC
9+roY7lekClNkRuiovLRQ3cypk/+vpOJh2LP8sKX7b5HZk7GV8p8Zpkv0Ri6QnGyU2ATIosjyDEUkQ04jpDmpi//i9x0NGhFvrWs
wIxjcSfjlZxiu66m0yyI7JF1oBM/vCIEGtpZ0eke6U0FZ5ovZ1+7HXSPqB7i1M8B/UiCV259kQR3sh6h1I63AiJbSTDLvF9kH5Pg
C1mWBPu96H2qbCXBkSEdABTsBSDzmHU7Xav7GsqhFskTcuoBAYRDVehAHBdOQ5w0oIoOqxYpBSDLYRjt1yWcRBGkkQYAMo3cN0WI
p7+OFxE5xBDigJAQngFtFiMIUl9VIFRDQrhhcwgqkFrPTkwrzgWa8S077NAjniHW1q2DoMVM4bJZjMysyxCy7GScW0xyi0lYjBhC
qRiPpIqEF03P/KgTUTRormD9oGCtHFqEUCq9tlQd2qr7G+RloSmEdPffDvebpjScTzyNzlVQ9lEzVqNnp6olknxHvGap4hlKc+io
XtZ7abNGXan1GLJn5Og1O/KWQHekpciqSwdA1uNTG4BkQUg6DUggFKmaAFIj65ogyT2zbei0tvwP7Tfv3LEVJLmpWxci8y26Dbib
3eo20Imm8yle0+dxN4LIfHxgqWd+0a0iZDifBhHnA200nA/DNXYTLLVfL7JZg9A0O5aO/B2KxaNiN05WkfMiDEm6ITkvw1BPheeS
Hpn9XA2vqplBiN1WeJWkxiKRzCtI/FwdQz09svq5GEN+LhgDNIschiE/35upubzISVQ4b4L7qVA4QHVCiiG74TjwEmLuZxDidL/m
rAxDdrtTIqE5s4YhZwZT2Xxzp0ee6SLxDK87kiPPfEGJIWfCSPTuj4FNMOT7wbxAMzFkhhxufrjKqzDGqzgyA2HIvXrgDT2CoPDz
BT0hw9C8LLCDMnkCLxjyDA6Daz5hJ9Qw5LywGKyzfsDCz8xgiTb8+ZPoUDjdULyMwUYR5wWvTV7FO9xwZoaOMwNrlGAKIUtXxWsA
H7lydt9ydV6MIedFsCiJ+hEGCjdON4zSkvGGlq6KqhMbxZyX4FXOS6BRzHnBBOBvkp5YufPLY+sAOS8cDt19AztbVL/YsUfuhyMt
kHjmjYY39OdPw74hXt1hj5JSU4h8Q3hkIRe+YMiDqGHIsiP7Uys5srAL3zFUU22wpcJLrqhZNmTakDTpiUTZClepJz18Lk2Tnszk
kOReUeeFdVidF2OIs9wrtabaqK5DvGFzHSqG3F6YV0sdW5qlHmUly/Ni4YdQ8+Z+iIW3uDjgqp7z6n4ubOVeUxcNCGuju3rxq2/k
6h0uBnaAkZtyvqgbfspq5A2oQ428ATfUPG9oScNBI6VAKyvlvCjnRa5DaEqNlAIdQCOlJKtSHSrn5/IWA3YAlQhzvMrDHBYVKpaK
oalvqHKW9FRTP1RN7xRv0CeO7U2phqtvrTmvqDeg965+FV4V9QYWo3kDDIvRLG1b6EwB3gfpsHkSjRpCPZxaot9u6MUUPSbGpq5s
V9h4yML78JToFi0ZiVlF77XWZ+58J5oP1H2WjnZa3R0wzjhR+aSF0YT1TKUxRn6bnZ6Jekws38csJ6pOLla1t/HJiWosPdS36eeZ
KsZE9j7dPlPV/cuF0++rdV8REhX+0QoFbezv9lnsgTnwmayRvzJCd2XUefEpJIubCH1IcSLz16LBoeSZjEO2dmGN9ZZpMbss1njM
KwWRxTMEDJDPVPEUsDCtyaxciZAzsQ1sEK+r3bHps/d7SfyNIJEVneAjmRey5ds1xQ5ATPfkIkpPRFmUPolWlBqOrCfR5yDdqdZB
3gfHZ6qPUfqkWlEKBrAnqs/xt1N5wu5gJn+iWjM2uSSTeMyB0f2ZLEKzZ1Gyk7X1XLs450VoPskeoRkj+TGLBxtVERl7odovyeRb
Cllv/VYP3wEghazHfQNfRp3JZmx+w1gXkfoki28ZIJAFForGfi/1XuQRdJYhhNbE8HZrAHkEHlpk4fwDQiFeR0jEVWcAeQOS9hH7
CVpf5kDEowbKsAbVaM2KIXTaddcp3G4NnBHSDqPBMxIBAXeLBgjabUUH0o83JZFZOQp6dNDV7utIANZ0UdS2kFE8bhvcblaieEnN
TmOZblZniyBSX0qRL8QSY8c1in2Rx3JTaIYRH3wYFH3EGNFgIMUdhFZ5b4hy4BTMx0krHd3kDMAl9KgJAcRrOwS9ZYBDF+c1Axwa
BlGgI0jXp5JIdF0bMoDWZywNbVjjW9QCVzW/ASDS3ccQEq48IBJvRYjUdE00ThASXg5lixbN8do4TognAte0dDc/qSKkR9BCJD1p
NGcgn+jNQCuMY0ePXybE9ZTyjxPiuj8az0h3hKF/R88OQzVf5U2Khlj5dZCtimHPwJDz6gohy45FnPPiVE0+Bk43DF4wEUjOazVf
4IaS61Dyc0U/Fwsf/VysqOjndryhpZqPfi7esOY6rDmvmvNqOa8m6Yat5hsGL7jKShZbPgdu8PecUXwzgsXr5fA5Cb9OgTPxes6r
57xG7oQjd8JIDNiOkRmgGF46JcLzasviVd4CLnBVtGVhXmfyiYhhKLreGLLsClmdCMGQZMl9PZKwGDEGhhmP45NLmELXGLjiVTW7
TdcYGG+oJZVQOZVQa75hfq6a86rOy/CqeGjiVZZuuN7ECqHgBZGayt5yFa5SGUPeNMMOFWV0skrSYor97UZJFPXyUt2eJrreMIGQ
HvvG/Dq3zbB4FmNuEv9qBbk9+1lwR//UkFMgaq7/A6E3AuRpNwAA"""

def _load(b64):
    raw = gzip.decompress(base64.b64decode(b64))
    d = pd.read_csv(io.BytesIO(raw))
    d["Date"] = pd.to_datetime(d["Date"], format="mixed", errors="coerce")
    d["year"] = d["Date"].dt.year
    return d

sum_ = _load(_SUM)   # RMSUM1..4  -> XS1 (a), XS2 (b)
sd   = _load(_SD)    # RMSD1, RMSD2 -> XS3 (c), XS4 (d)
print("sum_ years:", {s: sorted(int(y) for y in sum_[sum_.Station_ID==s].year.dropna().unique())
                      for s in sum_.Station_ID.unique()})
print("sd   years:", {s: sorted(int(y) for y in sd[sd.Station_ID==s].year.dropna().unique())
                      for s in sd.Station_ID.unique()})



import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

import matplotlib as _mpl
_mpl.rcParams.update(_RC_DEFAULTS)   # clear any style left by an earlier cell
plt.rcParams.update({
    "font.size": 10.0,
    "font.family": "DejaVu Sans",
    "axes.labelsize": 10.5,
    "axes.titlesize": 11.5,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.fontsize": 8.7,
    "axes.linewidth": 0.9
})

# Profile colors
GRAY = "0.55"
ORANGE = "#E8943A"
RED = "#C0392B"
PURPLE = "#7B2D8E"

# Era shading colors
SH_PRE = "#D6E6F2"
SH_EARLY = "#FBE3C8"
SH_MECH = "#F2D4D0"

# Era boundaries placed between calendar years
PRE_END = 2011.5
EARLY_END = 2016.5


def prof(df, station, year):
    """Return distance and elevation arrays for one station-year survey."""
    s = df[
        (df["Station_ID"] == station) &
        (df["year"] == year)
    ].sort_values("Distance")

    return s["Distance"].values, s["WL"].values


# Draw directly at the final Word frame: 170 mm wide, with the existing
# manuscript aspect ratio (6120130 x 7442200 EMU). No post-generation shrinking
# or stretching is needed in Word.
FIG_W_IN = 170.0 / 25.4
FIG_H_IN = FIG_W_IN * (7442200 / 6120130)
fig = plt.figure(figsize=(FIG_W_IN, FIG_H_IN))

# Reduced spacing addresses the request to bring panels closer together
gs = fig.add_gridspec(
    3,
    2,
    height_ratios=[1.0, 1.0, 1.05],
    hspace=0.42,
    wspace=0.22,
    left=0.105,
    right=0.985,
    top=0.965,
    bottom=0.070
)


def panel_label(ax, text):
    """Use a regular-weight, left-aligned panel title."""
    ax.set_title(
        text,
        loc="left",
        pad=4,
        fontweight="normal",
        fontsize=11.5
    )


def xsec_panel(
    ax,
    df,
    station,
    panel_text,
    pre_years,
    recent_year,
    recent_color,
    t_2018,
    t_recent
):
    """Plot one repeat cross-section panel."""
    plotted_x = []

    # Earlier surveys (some profiles extend into early mining)
    for i, year in enumerate(pre_years):
        x, z = prof(df, station, year)
        plotted_x.append(np.asarray(x, dtype=float))

        ax.plot(
            x,
            z,
            color=GRAY,
            lw=1.2,
            alpha=0.85,
            label=(
                f"Earlier surveys ({', '.join(str(y) for y in pre_years)})" if i == 0 else None
            ),
            zorder=1
        )

        # Jim requested thalweg markers for every earlier survey profile.
        ti_pre = np.nanargmin(z)
        ax.scatter(
            x[ti_pre],
            z[ti_pre],
            marker="v",
            s=85,
            facecolor="white",
            edgecolor=GRAY,
            linewidth=1.0,
            zorder=4
        )

    # 2018 profile
    x18, z18 = prof(df, station, 2018)
    plotted_x.append(np.asarray(x18, dtype=float))

    ax.plot(
        x18,
        z18,
        color=ORANGE,
        lw=2.2,
        label=f"2018 (thalweg = {t_2018:.1f} m)",
        zorder=2
    )

    # Most recent profile
    xr, zr = prof(df, station, recent_year)
    plotted_x.append(np.asarray(xr, dtype=float))

    ax.plot(
        xr,
        zr,
        color=recent_color,
        lw=2.4,
        label=f"{recent_year} (thalweg = {t_recent:.1f} m)",
        zorder=2
    )

    # Thalweg markers
    ti18 = np.argmin(z18)

    ax.scatter(
        x18[ti18],
        z18[ti18],
        marker="v",
        s=150,
        color=ORANGE,
        edgecolor="black",
        linewidth=0.8,
        zorder=5
    )

    tir = np.argmin(zr)

    ax.scatter(
        xr[tir],
        zr[tir],
        marker="v",
        s=150,
        color=recent_color,
        edgecolor="black",
        linewidth=0.8,
        zorder=5
    )

    # Set limits from the actual profiles with only 1% padding. This is
    # especially important for RMSD1 (panel c), whose data end near 527 m.
    finite_x = np.concatenate([x[np.isfinite(x)] for x in plotted_x])
    xmin, xmax = float(finite_x.min()), float(finite_x.max())
    xpad = 0.01 * (xmax - xmin) if xmax > xmin else 1.0
    ax.set_xlim(xmin - xpad, xmax + xpad)

    ax.set_xlabel("Distance from left bank (m)")
    # A shared unit is sufficient across each row. Suppressing the repeated
    # right-column title prevents it from intruding into the left panel.
    if ax.get_subplotspec().colspan.start == 0:
        ax.set_ylabel("Elevation (m PWD)")
    else:
        ax.set_ylabel("")

    # Add vertical headroom so the legend does not cover a profile.
    y0, y1 = ax.get_ylim()
    ax.set_ylim(y0, y1 + 0.43 * (y1 - y0))

    ax.legend(
        fontsize=8.4,
        framealpha=0.96,
        loc="upper right",
        bbox_to_anchor=(0.995, 0.975),
        borderaxespad=0.2,
        handlelength=2.0,
        handletextpad=0.45,
        labelspacing=0.25
    )

    ax.grid(True, alpha=0.25)
    panel_label(ax, panel_text)


# Panels (a)–(d)
# Internal data IDs remain unchanged, but all displayed labels use XS terminology.

xsec_panel(
    fig.add_subplot(gs[0, 0]),
    sum_,
    "RMSUM1",
    "(a) XS1: upstream",
    [2008, 2010, 2014],
    2023,
    RED,
    10.7,
    10.6
)

xsec_panel(
    fig.add_subplot(gs[0, 1]),
    sum_,
    "RMSUM2",
    "(b) XS2: mined reach",
    [2008, 2010, 2014],
    2023,
    RED,
    8.6,
    8.6
)

xsec_panel(
    fig.add_subplot(gs[1, 0]),
    sd,
    "RMSD1",
    "(c) XS3: 5 km downstream",
    [2010, 2013],
    2024,
    PURPLE,
    7.2,
    -1.1
)

xsec_panel(
    fig.add_subplot(gs[1, 1]),
    sd,
    "RMSD2",
    "(d) XS4: 6.5 km downstream",
    [2010, 2013],
    2024,
    PURPLE,
    6.7,
    5.6
)


# ---------------------------------------------------------------------
# Panel (e): thalweg evolution
# ---------------------------------------------------------------------

axe = fig.add_subplot(gs[2, :])


def thal(df, station, years):
    """Calculate thalweg elevation for each requested survey year."""
    return {
        year: prof(df, station, year)[1].min()
        for year in years
    }


series = {
    "XS1: upstream": (
        thal(sum_, "RMSUM1", [2008, 2010, 2014, 2018, 2023]),
        "#1f77b4",
        "o",
        "+2.3"
    ),
    "XS2: mined reach": (
        thal(sum_, "RMSUM2", [2008, 2010, 2014, 2018, 2023]),
        "#2ca02c",
        "s",
        "-0.4"
    ),
    "XS3: 5 km downstream": (
        thal(sd, "RMSD1", [2010, 2013, 2018, 2024]),
        "#d62728",
        "^",
        "-10.1"
    ),
    "XS4: 6.5 km downstream": (
        thal(sd, "RMSD2", [2010, 2013, 2018, 2024]),
        PURPLE,
        "D",
        "-1.7"
    )
}

XR0 = 2005.5
XR1 = 2027.8

# Three mining-era bands
axe.axvspan(
    XR0,
    PRE_END,
    color=SH_PRE,
    alpha=0.65,
    zorder=0
)

axe.axvspan(
    PRE_END,
    EARLY_END,
    color=SH_EARLY,
    alpha=0.65,
    zorder=0
)

axe.axvspan(
    EARLY_END,
    XR1,
    color=SH_MECH,
    alpha=0.65,
    zorder=0
)

# Era boundaries
axe.axvline(
    PRE_END,
    color="0.45",
    linestyle=":",
    linewidth=1.2,
    zorder=1
)

axe.axvline(
    EARLY_END,
    color="0.45",
    linestyle=":",
    linewidth=1.2,
    zorder=1
)

# Era labels centered within each band
ERA_LABEL_Y = 12.4

axe.text(
    (XR0 + PRE_END) / 2,
    ERA_LABEL_Y,
    "Pre-mining",
    ha="center",
    va="center",
    fontstyle="italic",
    color="0.35",
    fontsize=9.6
)

axe.text(
    (PRE_END + EARLY_END) / 2,
    ERA_LABEL_Y,
    "Early mining",
    ha="center",
    va="center",
    fontstyle="italic",
    color="0.35",
    fontsize=9.6
)

axe.text(
    (EARLY_END + XR1) / 2,
    ERA_LABEL_Y,
    "Mechanized mining",
    ha="center",
    va="center",
    fontstyle="italic",
    color="0.35",
    fontsize=9.6
)

# Plot thalweg series
for label, (values_by_year, color, marker, delta) in series.items():
    years = sorted(values_by_year)
    values = [values_by_year[year] for year in years]

    axe.plot(
        years,
        values,
        color=color,
        marker=marker,
        markersize=10,
        linewidth=2,
        label=label,
        zorder=3,
        markeredgecolor="black",
        markeredgewidth=0.4
    )

    # Keep the longer downstream-change labels inside panel (e) and clear
    # of their connecting series.  The XS3 label is placed after its final
    # point so the descending red line cannot pass through the text.
    if delta == "-10.1":
        label_offset, label_ha = (7, 15), "left"
    elif delta == "-1.7":
        label_offset, label_ha = (7, -1), "left"
    else:
        label_offset, label_ha = (7, 0), "left"
    axe.annotate(
        (f"Δ =\n{delta} m" if delta in {"-10.1", "-1.7"} else f"Δ = {delta} m"),
        xy=(years[-1], values[-1]),
        xytext=label_offset,
        textcoords="offset points",
        ha=label_ha,
        va="center",
        fontweight="bold",
        color=color,
        fontsize=9.5
    )

axe.set_xlim(XR0, XR1)
axe.set_ylim(-2, 15.5)

axe.set_xlabel("Year")
axe.set_ylabel("Thalweg elevation (m PWD)")

axe.set_xticks([2008, 2012, 2016, 2020, 2024])

axe.legend(
    loc="lower left",
    ncol=2,
    fontsize=8.7,
    framealpha=0.92
)

axe.grid(True, alpha=0.25)
panel_label(axe, "(e) Thalweg elevation through time")

fig.savefig(
    "fig5_cross_sections_final.png",
    dpi=300,
    facecolor="white"
)
fig.savefig(
    "fig5_cross_sections_final.pdf",
    facecolor="white"
)

plt.close(fig)

print(
    "Thalweg deltas:",
    {
        station: round(
            list(values[0].values())[-1] -
            list(values[0].values())[0],
            2
        )
        for station, values in series.items()
    }
)

print("Saved: fig5_cross_sections_final.png and fig5_cross_sections_final.pdf")


## Figure 6: water-level decline

The panel (b) and (c) y-axis labels are now on two lines each; at one line
they met in the left margin. Retains the annual stage-range series that the
Methods, Results and caption all refer to.


In [ ]:
# @title Figure 6 - water-level decline
import pandas as pd, numpy as np, gzip, base64, io
import matplotlib.pyplot as plt, matplotlib.dates as mdates
from matplotlib.ticker import MultipleLocator


BLOB='''...'''
df=pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(BLOB))),index_col=0,parse_dates=True)
d262=df[['WLu']].rename(columns={'WLu':'wl'}).dropna()
d263=df[['WLd']].rename(columns={'WLd':'wl'}).dropna()
print(d262.shape,d263.shape)


def dry_annual(daily):
    df=daily.copy(); df['year']=df.index.year; df['mon']=df.index.month
    dry=df[df['mon'].isin([2,3,4])]; g=dry.groupby('year')['wl'].agg(['mean','count'])
    return g[g['count']>=60]['mean']
dryU=dry_annual(d262); dryD=dry_annual(d263)

def annual_range(daily):
    """Annual maximum minus minimum of the daily record."""
    values=daily.copy(); values['year']=values.index.year
    g=values.groupby('year')['wl'].agg(['min','max','count'])
    return (g['max']-g['min']).where(g['count']>=180)

rangeU=annual_range(d262)
rangeD=annual_range(d263)
# The WLu record has a prolonged 1998–1999 gap, so 1999 is not plotted.
rangeU.loc[1999]=np.nan

def era_mean(s): return (s[(s.index>=1988)&(s.index<=2011)].mean(),s[(s.index>=2012)&(s.index<=2016)].mean(),s[(s.index>=2017)&(s.index<=2025)].mean())
yrs=sorted(set(dryU.index)&set(dryD.index)); slope=pd.Series({y:(dryU[y]-dryD[y])/6.5 for y in yrs}); base=slope[slope.index<=2011].mean()
print('WLd drop',round(era_mean(dryD)[2]-era_mean(dryD)[0],2),'| WLu drop',round(era_mean(dryU)[2]-era_mean(dryU)[0],2))
print('slope',round(base,3),round(slope[2019],3),round(slope[2024],3),round(slope[2025],3))


CU='#2166ac'; CD='#b2182b'; CSL='#333333'; B1,B2=2011.5,2016.5
X0, X1 = 1987.5, 2025.8
MAJOR_YEARS = np.arange(1990, 2026, 5)
def shade(ax,b1,b2,x0,x1):
    ax.axvspan(x0,b1,color='#4393c3',alpha=0.10,zorder=0)
    ax.axvspan(b1,b2,color='#fdae61',alpha=0.18,zorder=0)
    ax.axvspan(b2,x1,color='#d6604d',alpha=0.14,zorder=0)
def panel_letter(ax,ch):
    ax.text(0.992,0.95,ch,transform=ax.transAxes,ha='right',va='top',
            fontsize=11.5,fontweight='normal',
            bbox=dict(boxstyle='round,pad=0.16',fc='white',ec='none',alpha=0.80))

import matplotlib as _mpl
_mpl.rcParams.update(_RC_DEFAULTS)   # clear any style left by an earlier cell
plt.rcParams.update({
    'font.family':'DejaVu Sans',
    'font.size':10.0,
    'axes.labelsize':10.5,
    'axes.titlesize':11.5,
    'xtick.labelsize':9.5,
    'ytick.labelsize':9.5,
    'legend.fontsize':8.7,
    'axes.linewidth':0.9
})
# Draw directly at the final Word frame: 170 mm wide, with the existing
# manuscript aspect ratio (6120130 x 5056505 EMU).
FIG_W_IN = 170.0 / 25.4
FIG_H_IN = FIG_W_IN * (5056505 / 6120130)
fig,(axa,axb,axc)=plt.subplots(3,1,figsize=(FIG_W_IN,FIG_H_IN))

# ---- (a) ----
axa.plot(d262.index,d262['wl'],color=CU,lw=0.75,alpha=0.85,label='WLu (upstream)')
axa.plot(d263.index,d263['wl'],color=CD,lw=0.75,alpha=0.85,label='WLd (downstream)')
axa.set_xlim(pd.Timestamp('1987-07-02'),pd.Timestamp('2025-10-20')); axa.set_ylim(6.5,18)
shade(axa,pd.Timestamp('2011-07-01'),pd.Timestamp('2016-07-01'),pd.Timestamp('1987-07-02'),pd.Timestamp('2025-10-20'))
axa.axvspan(pd.Timestamp('1998-10-01'),pd.Timestamp('1999-05-01'),color='0.55',alpha=0.20)
axa.text(pd.Timestamp('1999-01-15'),14.25,'WLu gap',color=CU,fontsize=8.8,
         fontweight='normal',ha='center',va='center',rotation=90,
         bbox=dict(facecolor='white',edgecolor='none',alpha=0.78,pad=0.8))
for x,t,c in [(pd.Timestamp('2000-01-01'),'Pre-mining',CU),
              (pd.Timestamp('2014-01-01'),'Early\nmining','#b8860b'),
              (pd.Timestamp('2020-07-01'),'Mechanized\nmining',CD)]:
    axa.text(x,17.85,t,color=c,fontsize=9.0,fontweight='normal',ha='center',va='top',
             linespacing=0.88,
             bbox=dict(facecolor='white',edgecolor='none',alpha=0.62,pad=0.5))
axa.set_ylabel('Water level (m PWD)')
leg_a=axa.legend(loc='lower center',bbox_to_anchor=(0.50,0.015),ncol=2,
                 framealpha=0.94,handlelength=3.0,handletextpad=0.55)
for line in leg_a.get_lines():
    line.set_linewidth(2.2)
axa.set_xticks([pd.Timestamp(f'{year}-01-01') for year in MAJOR_YEARS])
axa.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axa.xaxis.set_minor_locator(mdates.YearLocator(1))
axa.grid(alpha=0.25); panel_letter(axa,'(a)')

# ---- (b) ----
axb.plot(dryU.index,dryU.values,'o-',color=CU,ms=6,lw=1.5,label='WLu (upstream)')
axb.plot(dryD.index,dryD.values,'s-',color=CD,ms=6,lw=1.5,label='WLd (downstream)')
axb.set_xlim(X0,X1); axb.set_ylim(6.7,13.3); shade(axb,B1,B2,X0,X1)
pU,pD=era_mean(dryU)[0],era_mean(dryD)[0]
axb.hlines(pU,1988,2011,color=CU,ls=':',lw=1.2,alpha=0.8); axb.hlines(pD,1988,2011,color=CD,ls=':',lw=1.2,alpha=0.8)
axb.set_ylabel('Dry-season mean\nwater level (m PWD)')
axb.grid(alpha=0.25)
axb.yaxis.set_major_locator(MultipleLocator(1))
axb.set_xticks(MAJOR_YEARS)
axb.xaxis.set_minor_locator(MultipleLocator(1))

# Annual stage range requested in Jim's caption and Results.
axbr=axb.twinx()
axbr.plot(rangeU.index,rangeU.values,'o--',color=CU,ms=4.8,lw=1.25,
          markerfacecolor='white',markeredgewidth=1.0,alpha=0.70,label='WLu annual range')
axbr.plot(rangeD.index,rangeD.values,'s--',color=CD,ms=4.8,lw=1.25,
          markerfacecolor='white',markeredgewidth=1.0,alpha=0.70,label='WLd annual range')
axbr.set_ylim(0,7)
axbr.set_ylabel('Annual stage range (m)')
axbr.yaxis.set_major_locator(MultipleLocator(1))

h_mean,l_mean=axb.get_legend_handles_labels()
h_range,l_range=axbr.get_legend_handles_labels()
axb.legend(h_mean+h_range,l_mean+l_range,loc='lower left',ncol=2,
           framealpha=0.92,handlelength=2.5,columnspacing=1.2)
panel_letter(axb,'(b)')

# ---- (c) ----
axc.plot(slope.index,slope.values,'o-',color=CSL,ms=5,lw=1.5)
axc.fill_between(slope.index,0,slope.values,color=CSL,alpha=0.12)
axc.set_xlim(X0,X1); axc.set_ylim(0,0.79); shade(axc,B1,B2,X0,X1)
axc.axhline(base,color=CU,ls='--',lw=1.5,alpha=0.9)
axc.text(1998.6,base+0.015,f'Pre-mining mean:\n{base:.2f} m km$^{{-1}}$',
         color=CU,fontsize=8.8,va='bottom',ha='center',linespacing=0.95,
         bbox=dict(facecolor='white',edgecolor='none',alpha=0.88,pad=0.9))
axc.annotate(f'Peak {slope[2019]:.2f} m km$^{{-1}}$\n(2019)',xy=(2019,slope[2019]),xytext=(2013.0,0.735),
             color=CD,fontsize=8.8,fontweight='bold',va='top',linespacing=0.95,
             bbox=dict(facecolor='white',edgecolor='none',alpha=0.78,pad=0.7),
             arrowprops=dict(arrowstyle='->',color=CD,lw=1.0,shrinkA=2,shrinkB=2))
axc.annotate('Slope declines\nas WLu drops',xy=(2024,slope[2024]),xytext=(2017.1,0.075),
             color='gray',fontsize=8.8,ha='center',va='bottom',
             bbox=dict(facecolor='white',edgecolor='none',alpha=0.82,pad=0.8),
             arrowprops=dict(arrowstyle='->',color='gray',lw=1.0,shrinkA=3,shrinkB=2))
axc.set_ylabel('Water surface slope\n(m km$^{-1}$)'); axc.set_xlabel('Year')
axc.set_xticks(MAJOR_YEARS)
axc.xaxis.set_minor_locator(MultipleLocator(1))
axc.grid(alpha=0.25); panel_letter(axc,'(c)')

fig.subplots_adjust(hspace=0.14,top=0.985,bottom=0.090,left=0.105,right=0.895)
_png = _io.BytesIO()
fig.savefig(_png,format='png',dpi=300,facecolor='white')
with open('fig6_water_level_final.png','wb') as _fh:
    _fh.write(_png.getvalue())
fig.savefig('fig6_water_level_final.pdf',facecolor='white')
plt.close(fig)

print('Saved: fig6_water_level_final.png and fig6_water_level_final.pdf')


## Figure 7: groundwater response at well GWn

Three mining-era symbols in panel (b), the 2013 analytical split labelled
as such, and the four panel (a) annotations moved into bands the weekly
series leaves clear.


In [ ]:
# @title Figure 7 - groundwater response
"""Figure 7 v3 - groundwater response at well GWn.

Changes from cl_fig_gwt_v2, from Jim's review:
  243, 245  panel (b) no longer colours points by year through a colorbar. The
            three mining eras are now shown as three marker shapes in three
            colourblind-safe colours, and the colorbar is gone.
  261       type scale enlarged and made consistent with the other figures.

Panel (a) follows the manuscript's 2013 groundwater-response break and its
Nov-Apr / May-Oct groundwater-season definitions.
"""
import json, gzip, base64, sys
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch
from scipy import stats

# ---------------------------------------------------------------- chat 31
# FIG_SCALE rescales the drawing canvas while the font sizes below are left as
# authored. AGU prints a two-column figure at 170 mm (6.693 in), so text set at
# F pt on a canvas W inches wide reaches the page at F * 6.693 / W pt. Scaling
# the canvas down is therefore the same as enlarging the type on the printed
# page, and it preserves every layout decision in relative terms.
FIG_SCALE_W, FIG_SCALE_H = 0.995, 0.995
# ---------------------------------------------------------------------------
_STYLE_W, _STYLE_H = 11.0, 9.6
FIG_W, FIG_H = _STYLE_W * FIG_SCALE_W, _STYLE_H * FIG_SCALE_H
FS = apply(_STYLE_W)

GWT_B64 = "H4sIAHCPWmoC/22dTc7DPI6E79Lr/l6Y1C/PMugbNGY395/YJMUqJ0BWDyTZliVKZNHK//zPv8T2/ueSf67+r3/r3+z/+XcxkYctYvub6Xgz/eAfbH8z+VFX9Qezf/37f//vv/8t1v651jeT/q7b/lF5P9uH7TfrP/qgZx8we+oOQxZ9AOXG57Zv1o2YPWwgk/kwRNpupFS1XTcTYPPze5ggk6futZHpp+71NyexT/fJ39Bi6+mCTzlBdnfBhzVi+2ZrIru7QP4u6Pr9DIMP68T2wxSZ/Kh7DwP5W42Y3awDs0/XPM87kcnT9W0Qe7reEOmn+9pflZLrMzZuNBHJ9SAqJutmC5H2BxmyJjczqCrPsO1/Quwetv1Pid3Dtv81QvtG1Zuiz9P3vzGQ3U/f8Rn0efj+txHdD9/r3uwxA09NM2KfqqPe18PuITtqjD3sHrOjxuLD7jH7Ya2YPmP2zcTrLmTqdTexdTO4lRbNKTFvThsyb+7F1pv1ZwB8WEcm1w+2vtk9BEa9Rntm/GNBXmx/l7unALc3ninwxeweUh26ecYUmIh8BgxCRpPiYfcgkJo89hiB9rCG7O4BoUGwnh74MEGmjwExaq89BgTG1H7mgPy1huyeA1Ij+WH62A+qqo/5gAvYY7XkbyGSd0ULs311ZG62pa55W4Cn55SQ9xwiN9qNarrV3oKsPcYDOvi2AI/ZgZlyW4DH7jREer3e9M0WWix7LMBjd4Yik8fujE5sP2wiu5//w2KO2fXMdwELGOipmsuCMxloFZ3d4/VYxUDPcN1QLG3WJcjcZl1Uzo3WNZG51cJyLdvryKK9RszbU2TRXqEeRpDR8xS2kcn8Zvq8RVvIfAQgG8/4P6/Rke9ctiDzncvF7Bn/UHXy+Hfk4x9bm88AkGPwnfmibdDF63l+rrueDhAaAevpgJqwzvSxHZNYu163t58ZIH8bkTw1caDsZwZ87q4RW0+fFLLHYH1mMSK5YBIHWjCJHenT573YPdcFDGegDebVUexbDZnvW6HfbuabDqgrz3D9TH9FJs/MUSFmuDo502e161BOn8efZ4Pm7H7+SfenTwfUftnZ3QNv5puWkdeQxwIsNAoPkk4G4GEqaAAcPbZjabE0WWsjc5O1B7GnriFyi2XQXIvmbCDz5mwT8+YMmbItkme2y+uyPSzgQOQGUBHdIwB609ljOi4oN2Lbjjc8Ytu+EekziLsQWziw5Zn+PvwnMh//1ya2aK4/TDtORHmmuqCBcfRYjo7Id+wTkW/YjZCRAZNn7tOa68x3K7qJ+aI7kel894ilm9WJPeYK376FuSLkC/aFyL2sq656z/9Xn9/zv78e7J7rvtOjcu5n1r3JcbUHMnczjdB+jRuRGPxSz3DP/veYu5l9s/vx3+x+/je7OwCYPrN/wubEUexXiPmGpXVij+HIfdLNNHysjsh9rE7FhGzOg9zHOsuQPrOfNh2OfNOBKFZrRLFYD2S+WEPNHg8/EMn7AXo8uxJ67v+Cmx3x8g1RhBgIbYwSOPOtKlxgxlIlg5jvcon5XDViPlk7oWfSwBVWrNMdUazTiNxOXVTT7RQ8+w47pQOZ26lGaOGUeVCs01Sz0Tqtz6xfr560mKcLkU/TTejptF2N3ZN+8Ju6J/1jaawR8wV+IdP5mkIi72nqzL6Z/Kj7mqYP+5qm97Rf30z6N7t74IttYq2CgtaRhaUyYpsGesug4IdBOY119Wx8WgYFa960jAl+TIYgc8/ijID2zHLBielooy/kzJsbDZk3NxYxQ3PTnqnPPs7DhMyNo6fmomIeWtqFRpg9Qm721iLmFpOYT4ALUWPfoD2Tf9Hi9bBYqTay8CsGsY0Ttj2zn0zOg3wKXMwM1/0HKa217Zn9il6AIyMH4mG+SuPL3+FUXMTcqRB4VHtZqwe5tcJLWHgVm9Di/r3nf8dwqzMf/I3QRjP0ILfU9aQiYak3oU17wIf5UIUHlfSphcr5UMV70wwqEPN1ShHFOsXsGXHH0ejP3G/fzEdrhqOCLdwaPMgX6bPkP8x9Cqip4VL0gcx9iobIXYpBxdynmIVaOiiCzFvLQL2zaG4Ro8jIzXpGVToyYceopxjweYqNzG0fPtmIXUpvxOzVKSO2KYrItymXIPNdKjxYSgELkfCC21MJAHPQSwmA1laEFEyQRUxNiW3y0HsJAdBc6gBYLHUAW8hip0J13a2+BjH3K6Ccvf2KXjqAMnO/guqGX1HlQgmomd1TCYCte08lQMuE95QCarr3VAJgi9FTCYD1tacSoNgDoQSAWe8pBZS57ikFNOyAkAJwPIUU0LBPQguoeTdSCoAN/kgpAJamkVJADeKRSkAFLp299isjlYA3i631QOY2azAj52CkEgCr9UgloFbrkUJAO7pSsEUr+EghoB2Ny1ms10bsqbsFmZtAZCO2K1uJPdZjN2S+COyOzBeBF+PI6kghgMvNGAJ4LzPGwEXlfA7Ao6USsBC5ENAILdqJjNQBao0dKQNUXGGUDKCI5F3xiABUU9l4jJIBLkXmK/ZF5UIIIOZr9lXXPULAi7HhGSUFCCIPLHREbrLroqEDKL7B0AHwLYQOoPj8oQOAmRglBMB7OEIAPkNG1WQg8yXrbAtmCQFnCzBLCTjGY5YSQMV8vDYqprTazRQCYLLPFAJqVz9TB4Bd/UwdAGzCrDjgUTNmxQEXIY5IzgwDwmSfFQZci9imyOWsQCAh5Tk3KxCIbITJYuYmayuyewh07LuR0QUolpFAm8h8zdqEfMkiFi52oRWbVkYb8w2ceSj8GsgiFLiJkZswMxQoeG8eCRR8/Tv8i47I/QschhYOBiN7tZ9RQEPk7oURizBgtXbCgJch8+VaJrKIAwqxp+MasHSvOyI3V30RIw9jZhywAh0zw4AK4zyigPjmIwio+F4iCFgKpTMPg533vDIICKv1yijgm+n1g6038yjgm8n1g/2o+5It1jP72zf7bq/9aK/9bK/Jm/V3hOVh0r+Z/qj7irCsZ/6Pbyb6g9k306/3Md9RJmf2zeRHXW3frH313/rRB+tHH6wffbB+9MF+psGbifxgP+rqV//ZY7a+2P5m8qOu6g/27r/bFMxv9jUX7ml//WDv/hP5Hrsi32NX5HvsinyPXdHvPhD97gPR7z4Q/e4DeaffmacMDooGeHqg0vbAmfsbE5mv3rqKaTgcRwNy5nUNmZCv4tmB7YXaj9ZSFsC7ay+He5cuIFT1ZRB3CQPXRuaL9zWQucNxUd3Yv0G51Aa2IXOfexHyBBgq5i73gOZm+Nx9E9u0bu5SBxSRL994x6kOQGupDhgij7eZIIt4G1WNgBtcNfUB7OP90vF36QNKxXz3olSuyZulQtAQCa3CuxSCKcSeVXjVKD4SwSYWGgEhlwgUWThbddmQCGD3tlMigAV7p0QAe/+dEkHD/gyJAJy8nRLBZxwT8zCZGDIPk8FAkYy6nU29pUQArr6lRFA7eCuFYFAx33KfcWylEKxBbOP22kohOB1vpRBAa+2l41sJBCbIvDVm0ZoSszdLheAEtqwUghfzuh0ZSwRWEsGFSCjUZ6kQQKKJlUJAjfkEWFA1FYK1kYVGNpD5FFhCbOMW00ohmBuZz4A5iD3DeDZkPgUmXGLHFJiEDLe/lhoBROssNQLIArDUCD4MXpiFz43lLHzugch97rGILb65IxLMjsz38IuQ95wiC5Wguq5kgkFs8/sPmQAWD0uZAAyFpUwAhsJKJrgGMl+0BFHIBFRMK9Sv13WVSuAmIVlI+YQg1J/MQ2SEIvFoFlNMPEoUiUfE3F5tQvt4/o44QpjMH35sZPp+hIwQClTtsWAtRD5b3dAdVotOolivrNhJFG7EdsWvk0mpcon85QtVxRC5s8wTvgSZO9xG5ULRo3Ih6cGtZHxwCzIPEC4j9sT5FtX1COFqyDxEuKC9jBEuRPJVNYOEeNkMEsKbzRihIRLQJg7b3yxkjRoVJ0QohOALj2RusAi5vZpUNUKEUI5jhMkiVWwiw03mYevMa0cZI9SNzNdrmMQnRtgRhaqVA08qRNg7sY0TWyBXWJBFrjA1FyEyYBkj3A2ZQDj8MKs9QbJYYXexliu2IBNY7Q+DFTuZt2fwbD0cDZvEIF04mTsaC5Fv2gY118j0SCULN0SxZRvI3AReQmzXCuBsppsxkInQaJRMF9bwH5L5ogV3t2IKNGaGFk8yWVhr4EkmC2t4gcl8Dih05445oB1ZaJuIfM2WTWyVKOiM0oUTRbrkIrYqBJhMOxlQyYRhwT6OjGHBgRIpw4KDLFKGBQdjpAwzkwiUGiKUNg8zXJAkE4aV7iTz7y5ivmpfRmy9RplkBp4iauArfJhWvnBD5GO2UzEU9w+jdVsrX3gikq+ab6OllS6M95YygRALnYTZrtSuZD4BjkXR1AkUb7hnxvhA5uv2IOYLd2cGn+k4y4ThhigmwEamPHkUEoaBpU6gHdlrCmgJBdgrKRRcUDeVgqsR2zRVFKQCRRZTYBHjKaCgFWxkorT/0FILdkfmexdsz0jdO8zIWmgKBmVotNKGW0MWvkZVPYIBofg+FZG+hs9RC1Zd9KgFayALV2MT28cRTBSDFppLvWAPYi4YLGT++NuQ+fMbsQiQZXut0obPjrZV3rBSucgbZrbeLO3WQhSCPBULQR4R6vHOWqbjKTJvzgit+kAgmfb6kChZk/rgyFmPTz2Z+aee2F4PM2ib2KY9SUu5gDpgxNalD2L27uMRWxe87ETN6CDDtbGlViA1yFpqBfJqrT3fGBt03orNO3bUit17n8hUaG1sqRWAA9NSK6jJ2FIqKNPYUilQ6pP92rm0FAryY4LDdn0RlMytVl/IVHFqt9QJNFTqh4VOAAa0pU4gfxNRJGUJsYWWoqVKgO8rRAKYxi1FAoVBHBqB4ruRjLXhRTPWBgNWMth2bWSRlKjEPCkx6/bSCI5V6KURvJjXnch8xCJLjUAWMat04WS+2R5UznfbE9rLCOEWYrvyapJFwNGQRYhwE3u+MITmen6cOJGFv2HEDOd2rwDhgOYyQtg2MrdaYsRoEPQKEc6BzGOEHVjGCLFDM0aoVC50bkKbFtBeMULpyHzhFiFmt/3YVDc+T4ZymUjcCBluj3vFCBWRL1t4cxkihPYzQnh1ZCFrbGSvSdtfecQPihAh3ltECCsu0TNCKPD2I0BI1zwBQngNJ0A4qZwMfvsnj/hSYpAG5owDhMlC1drIfNEWJbbQORgQIUTEAcIBAcKNTGnTPypAeIbIqAChEYs1lpgfp3BtYhun68g84l6DZGQecQ/FKVk0x2xTaGFUHvEm5s7GNGL79bSZRjzgVjKNuBEyejmjjhQhFJtWQRajH66QecT4YDOG/0Lkot6kqp5HPOGyC79NThQBNyomkFiUTCnQOTKPGJyekXnEr2I+/IWK+Yqtkxi7WqPSiBWRsKswKosYu8nSzaiRfbKIBVFomkps1dcGydxeU9UGqUXOMou4KTLpZNZHZREvZpuiA6PSiC9isWYTMlrGR6UR553MSiM+FmtCGjGiEPWpWKQRU7kIj+1iabFGQxZRfWJ6oV8wM4sYDMVMkaDXq5gpEvRymGeKBL1i+DNVgl6e4UyVgMu5TMDt9XA0lOq6o4HlXCfotS+elUdMSMhXmJVFPAjx5mmWSrCJxYE6zFhhmKUSGFwjVQJTZK4SmBBbuFLOEgm4OfzwydnO1LyNDFPzkinPn5kqgdIItfQzFjJ3NPA9nlRiqhvrdl0jdAJ8tNAJpOLG832oSLIQCjayEArq9koomMhCKDBkkU2sxBYtIROUAkXmCxcMx6MUQPfJV9RtQTbxJAYpSckwJSlZfP1HdSOdeBVL0zWIuek6i+0CrWAji9g+XOMcLLKR+cEiQuX8YBGhcn6wiBbKg0WwW1wq6DWxVkoF73LavpprF+5eVkoFsCtZpRUYojimYRPbpIOulAoguLDqZJFLiW30wVcpBQ26KaWCLsSMVq8FWgHVjc37RhbfAMLrzpNF1kAGnywnUt75LDhZBHrKwnbhZS1sF6OFS/UqqQBe7DlbRDexTbuBVaeLcDl9d945XWTUU4RUAPZilVawmcHJJMlCLKgODbEAQgcLxIJNbJE/sEosgF7+Vgv2t1qwSy04fu4utaAZMYr571IL+kYmQl7CLrlgEIs99yyWgcKpyCLuSGhRKHJnoBB2/zsDhVyuRwcsRN4BS5DFvG3ENumoG+KEgizs4CBm7x4Yrz3czjghRIp2BQrxKWYuX4bsNW93RgqFXkZGCqchi0ghlfNIYevEHue5QbkdfkdH5JZrEiK3e2egUOjBLCS+vYlt0jJ2RQovKhfzlpBRIGZXpHAhitWbisXqLcQWefY7Q4UN70TS7xZCq9Jak8WkrUF2YoUyiG2Kz+2KFcpCFrHCTcwoYmUQKyTmg/YiRM6DVaSwF8pA4SRkNBet4oRG5XylvYC1WLlFiBmuyJbJxLCFt8wmhsXcMpsYlmmrbGJDFF8CbWThxDRi681Gxh0QRdhhENuvrhvvzZtlLnHDZ82DRjYVC4kTUQz/RcxexdY76mSZSwy7YaujRhqh9b7CSSVWZMJWxyqXWBCFy0HNNcHtu2UmsdDjW9grrJq5xFcjtklqtcolhhF2comlEWOJ0iqZWOpln2TiRsjQElmmEjdqTdLhQATfLjris0aS4VcgyfArkMNo2y9XJhL346o4uwfsyGcI9JycSaXu8TqP7iJX5hGXXx8Mg32OZHwhpZkuV6oE/YymYPubRXMbmdKeP5i92ZEJEIWnQcjQcDjjpCS5SiUYxASOkzwMN16OQiqiYo2UMrlSJeCXOF8DwBkPgGBfA2DRp1DJ8FOow+yb4adQzjZ9CnWYfTP5URc/hUqGn0I5M/oMKBk5G46Ugh3BKLVCLjhyRJGFvCnE4PDqZKHxF0qxwJhtTPNxJgN37s6URLpgBouAXCUWwEg+YkFDFAK/EFt1WNiHSYkFZ5xJqQVnekvJBenjOIutNrE4fnkV03SRJ7LQCzYypTBbsP1mRzDYyEIwaMi8vdOjkoJBP/qLSAkGtpGF96LE9vt5UzFYUC4VgynEDP16Z/BZVSIPwOpEFnkecNmZi7ci88V7DmSKjnOghVt8kZIMBrHYuzNjKyKpGTDb9DHcYfubyY+6qj+YvZm9rZekbPDFftR9WS9J2eDN5PrB1jfT/s3auw9COHgz6d9Mf9TVd/+FcPBmX30QysGbvfpASzoYhAw1fGeRWEvlfOTmXt9Zw9we0VQOYDnVVA5q1dUUDmDqaikHWiijhYR8CVchZt8sgoVQ90QLhRgFGp2xD+NMG+z+HfnjQyk6hThRhF0UmVLEJhivI1rRQuzNiR/DHITBQkdxDHehFU+/CNk5ED+RTNw1aQYK6dFXPDpU3O+FSzNM2Og1Z5xwElqorYlWSnFvyOJTEEILPRpnofIVO3HCi9nGuIQzTyleiBSVhkCG4VTRihIORIKBikD2VSq+2iqUEcKOiNOSgmFGiqP4ywQq1gRCYdIyPNjPNVuGB0fVbBkenH+MPp22zpRsGR1ctUa2jA7uM/5aBgdfyB0Ngwu0aO1CdDfGF/BU4oWteSYxXcATiQ3v1hOJrQZby0Riq/WxZSKxYRd5HrHVwGqZR7zPq2qZR7zrVbXMI17Uu55HvPC9zDh4WhH5PyVQKf+jBHyESX+U4Gyll8Vso0ltmUMMrl3LHOIy7S1TiBs9ws5PSRAJrTEtE4gbXjOPHYanyvThNohh8qSjOBuIikUmBiJOxJBW2cMNkUuaTZEp24pW+cPIhI5ZSCbyg+1v9tpVtUwgbvgK5Xyuv5AJBuccaaPtZ6v84cx1k175wxNRnOdFxUjXD7TwAyHplT1siFyGP/3UM3sY9uQ904c/Q7NB5Ra6fkZnnHmD0oktWsd7JRBTVc8fbmcP0CuBWDeycC4WMncudBDb7+uOjDIYMlFcQ3vpAlRK6dNJ6aULbLzETN9iT4QRZ2qKMM7d2Q2hz4TW4eIrpoKsjTCibZNKRiLxpQRdYmtwoX0+3SUY30HhE2U6sfaJME5cWFA9dYK5lWD878hAGJHSi2Ccycwl/VsIWfWYRy3Qhsw9rg5T6OgFG0ZkCQYNYCYX25oIQ+iDOzqigULPHdWgX3D1lA0GlkzdoC8q6TGIPhfC+ETQjKCH8U/1UdqBHQ9+lHhgi6FH0M7LHCkg3Jp+L6i5/z6jZqSI0CtCMVJE+LAxEXrYfwiUTBlh9k7waXLmZxcOvc01DKG3uc8kHKUlTLx6D8dmHGMyUk74lGwdYWQx7UHQbeAFj56KwmoTYWgKcxH0XMbdEeZfzcCF8gTzZYtgJKUJwpDEF1WPo0GtEfRVGN9RHlWiDCOtEd/wOa1kKkGP9gs80TnPHDskDyzp+N5TZhgyELrQQO89lYbWYMym1mD4mOdk80Ul42xzHGAWvppIXSj0hvtrXkEYH5MQcol4NYSRmA5vXY67ro2gVzeCkZ1vHaHGcZvUpu8K4BnPESZrToSCcpejOBIAnzAPBdhHZ5h1iAkhzhcItmm3MFN8qGjBLO1BmW1f3JH5Rva86VnqQzNibnKgXAYdpyDzfcEayHxfQFV9W4DFeogvC5E8NamUPz/eSI/nVyg3UixVYoZfQjmT+er2jDgSog+iZVa8cQ5ksTEkphftM2cGHCEoPjPgqHiJFRsBGcT8jwuJ3c9/0TjZ9LcMh9EnSc7wtO9kr7j7zHgjP8bJUiYWRw8xM9RVnel8XeP87WEjJtd3OXbiZ/3voVKxJrhznxltpBckwqlpjvQ98c7fHlY/nb89xHvTV2ZeMA+YLWSx2ucQWPXHh0rIKCa3MtYIgf2VscbK0XXW6rxORxlrPIN2VaxxE/P/KbyY1d8eOsq/PWyI/F8PGzFvrQ1i61U1c5Tx5jJHmYoJfRzhTClHUFYdddw7sY1ZWc7irxmonCqeUBHMIHNFVp10rIgEv40IZDhcVx1zDLd7jjkmJheG1lblJ29EnJ/srOEHibLqDJONyL0X7Mo8w2Qz2/ixhCzITh7IKLkp0H43Z/wZpCzITR7EKEPJGZ66lEzx6wZH7XpXlVd2h7PI0EekmFQXaOEXP7IqM9mICQv+KzOTO96c5nKdvbkzMRnm787EZFivdyYm15zemZfcq0t25iXDKrnrDw8nIjdWmcIdzOofWZP5H75dcCcuNIwKUexUGkaNp51Kwyjjt1NpGNQDrjSUedkpNHBzPf/0lZj/6euFyG2fwRUyL7l3ZL5facTgP28P2mRzdikNhkjo07BglKLkLP6XplCmJesgRqf4OKNTfBz5+B+KjD4plF0HmBgiwW/ZHb22K7vEhgnM3trgTrHhiy1M0XT2Wqx3qg0NRmeIDQ2rhthQW+ydYgN1eqgNDQdxqA0dR07IDfReJZ1zVWS+WteQCMGBZqdkKK9NYotnrGQkry9kHnk78XQDyaEjiw22IPMBexwUS9EBVnVL1QEmhaXqMP6E0Mal2VJ1gPC5peow/gh5Y6MTe1qbiLy1CVVddhh/x/Gy1B3e5fwfisdG5v9QjJd14WGUBbQUHkbtzSyVh14ZDAYHnQ9i9IWnWEoPNX0spYeGTzsjUkUV6atSsTq9RAex+pfVRPJs9Qcxj9gpIUrQE6u/PDREITxQMTHaIllKD406xDKlgxmFbC21h0bPZXlOKDH4M5kHhfZAxUJ8ABfBUnygOwnxge5YhLNRHb1WbEvtoeMdS3rYNcAkI3OD0GM5BhXzzfpE5Hv11ZD5epXBKr1SeejnYypnMVwXMh+uxmzhMqlXag+1nDrDPxU/zLfmG5niZNcr44jjDJ5gBpPdkS/YC5Gv1xuR/0f7hsZ6TFcj5u4FM3cvXmyhNKtXxg7fzC3WIrS/kBss7M0ZO7bNjNZ6Z3zikrM4uIOZoeOsVwYMzyruCP+hOJlHCxcz3O3rBaFCRHK9L7ozHR8RL9jO3LuGYpapHAOZ0IlBzhSdk0Ak+OmV0cE3E3318IkOdkQsFOlVoUFjZjj/nWUkHVkE0ollzLvYOyHZGWczOeNspmDcBVIJyZZxYIc+Zq/zOqRykq8MDjv0cXsUL5UMDX6mPJbUWGpTR3TmS622gdDXWs1Iv0rmJt9O/yD4NNnx4i3aPLqLQ29zWif4mJI94ZZ6OB2rC8LwOlKMDOi+gxF0v6NlaFMl05Q/cBF0u9hWI+gRkN0RRhZBp5LNX9GA+5yZ9DkGwogZr4Uwj2Ki6vF/cw2uvuJAsrkNoR9ssmQStDqJPVl8KH3Bxc9/JNog6B8KLyoZzji+4TzgxPZE6Kv7nnB1yz/0vQxh/vGMIHSnxK5BMHZk1R+RvXwfe90Qhj4EnRT5y7dKPhDGH3vDy5T8aHjBzZ8cZpyZkt75NZFl7v0g6HIbXEZzd4oFNdY7aRuhL3iixJbXTqiZyjyOK+PMR/aRsQP6yD4mRTOZeZxvzZ35wG7n9WhmM4/4R7ZknhvV9kCoT+2eaYIBt+/o4TotmlxTEN5tzr+rK0JvcxvDfZesvtQUGOafzolQnnwtecGn+qULoe8BDViqDLomQV9UVkcYQduM2jqMv4wY1GakiAqUTLXBlGDMlUUwHPhrEfQ8d4NbWjFXmiHzqSJ7E/Sv65VKxt+AI9z5idYiFn94vxH6TBkXlfSpso2ajKmy4SEtv7HHgWQZet1C0DcHXDI+L9WCR33oayCMxP/NMP7mcyGM9JzZEIYEAQVTg7BrIgy3/hKE4dfDVD06hMEDhRJxzzZF6OtiLWyaWsRdciOMdfFYuJZqxDjBjGBe+8yrlnrEqHHUUo+4J7Ug9E1+PzazpSQxSrF16GZvb2Ru9WpdaiVKbAHossRnTveGUDxPUBRhug6T4NMm3mUPS7rWQOimdOOt9zClIoLQTSne5chNkypBj70YQZ8stgn6ZBlGzB1Q7LcZca/aSLWUKe4NCkOjHBKHHvq6NsB15gqy+GNeE4IuuPWBMA6UESoZqWcD4M7vWrCHd26ZroEwDvEVKhnHSRyz11K4uPcNhjACAU0IegCeS8bnLb3aDPnic5/XJhj/jbkQRrL4NoTuXsxFMEICCE9MALpOTlAApqqcqIBRydglwEAMKeMeCx1hfPbYqWREBjeyCA2eBbSnnIGrf089Y1QahkN3CObxRnpKGrfrsAmG5ZKCafi6DoQ+uEdjaO46TIRu+E7+nfYKbO7VEXp/WN8EfZ/QJkLvELugTQ9vfkqe/XPP+Ob844Ly5HdfRgWjPzIfMaDHWvDRM8TZ10To1vQahvAeHxMcxp5RzjvOC7c0I9K9rSOMULcNgn7y1GUII9iJA2SlOHuMbK/Tml8w4iedYJzFs4ygYb6t9gx63lqbIMyo/yToH3ZMghFHMIAWk/3CO7KM/eGQtZjsNgj60ii7IfTJvrRKnuRrg3F8IqAyDWF81/eCbr5mXUhOREEXwogomBD0kMDVEfrg1gUlNXYKJ3MxoE9XNYS5VVgIvUMazMwIiH5G4mco/ef/AWcwv4IilgAA"
WL_B64 = "..."


def _unpack(b64):
    return json.loads(gzip.decompress(base64.b64decode(b64)).decode())


gw = pd.DataFrame(_unpack(GWT_B64), columns=['Date', 'gwt'])
gw['Date'] = pd.to_datetime(gw['Date']); gw['gwt'] = pd.to_numeric(gw['gwt'], errors='coerce')
wld = pd.DataFrame(_unpack(WL_B64), columns=['day', 'WaterLevel'])
wld['day'] = pd.to_datetime(wld['day'])
wld['WaterLevel'] = pd.to_numeric(wld['WaterLevel'], errors='coerce')

ERA = 2013
gw_raw = gw.copy(); gw_raw.loc[gw_raw['Date'].dt.year == 2013, 'gwt'] = np.nan
gw_agg = gw.copy()
m1 = (gw_agg['Date'] >= '2012-11-01') & (gw_agg['Date'] <= '2013-04-30')
m2 = (gw_agg['Date'] >= '2013-05-01') & (gw_agg['Date'] <= '2013-10-31')
gw_agg.loc[m1 | m2, 'gwt'] = np.nan
gw_agg['year'] = gw_agg['Date'].dt.year; gw_agg['month'] = gw_agg['Date'].dt.month


def dry_label(r):
    if r['month'] in (11, 12):
        return r['year'] + 1
    if r['month'] in (1, 2, 3, 4):
        return r['year']
    return np.nan


gw_agg['dry_yr'] = gw_agg.apply(dry_label, axis=1)
dry = gw_agg[gw_agg['month'].isin([11, 12, 1, 2, 3, 4])].groupby('dry_yr')['gwt'].mean().dropna()
mon = gw_agg[gw_agg['month'].isin([5, 6, 7, 8, 9, 10])].groupby('year')['gwt'].mean().dropna()
dry = dry[(dry.index >= 1988) & (dry.index <= 2024)]
mon = mon[(mon.index >= 1988) & (mon.index <= 2024)]


def seg(s, lo, hi):
    x = s[(s.index >= lo) & (s.index <= hi)]
    sl, ic, r, p, se = stats.linregress(x.index.values.astype(float), x.values)
    return sl, ic, sl * 10, p


d_pre, d_post = seg(dry, 1988, 2012), seg(dry, 2014, 2024)
m_pre, m_post = seg(mon, 1988, 2012), seg(mon, 2014, 2024)
wld['year'] = wld['day'].dt.year; wld['month'] = wld['day'].dt.month
wld['dry_yr'] = wld.apply(lambda r: (r['year'] + 1) if r['month'] in (11, 12)
                          else (r['year'] if r['month'] in (1, 2, 3, 4) else np.nan), axis=1)
wl_dry = wld[wld['month'].isin([11, 12, 1, 2, 3, 4])].groupby('dry_yr')['WaterLevel'].mean()
wl_dry = wl_dry[(wl_dry.index >= 1999) & (wl_dry.index <= 2024)]
coup = pd.DataFrame({'wl': wl_dry, 'gwt': dry}).dropna()
r_c, p_c = stats.pearsonr(coup['wl'], coup['gwt'])
print(f"dry pre {d_pre[2]:+.2f} (p={d_pre[3]:.2f}) | post {d_post[2]:+.2f} (p={d_post[3]:.1e})")
print(f"mon pre {m_pre[2]:+.2f} | post {m_post[2]:+.2f}")
print(f"phase r={r_c:.3f} p={p_c:.2e} n={len(coup)}")

BLUE, RED, GREEN = '#1f6fb4', '#c1392b', '#2e7d32'
PRE_BG, POST_BG = '#e8f0f0', '#fbeae6'
# Okabe-Ito, distinguishable in greyscale and to colourblind readers
ERA_STYLE = [('Pre-mining (\u22642011)', '#0072B2', 'o', 60),
             ('Early mining (2012\u20132016)', '#E69F00', '^', 78),
             ('Mechanized mining (2017\u20132024)', '#D55E00', 's', 66)]

fig = plt.figure(figsize=(FIG_W, FIG_H))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.15], hspace=0.16,
                      left=0.085, right=0.985, top=0.985, bottom=0.062)
chk = LayoutCheck(fig)
chk.ink_threshold = 185

# ------------------------------------------------------------------ (a)
ax = fig.add_subplot(gs[0])
kw = dict(zorder=0, lw=0, antialiased=False)
ax.axvspan(1988, ERA, color=PRE_BG, **kw)
ax.axvspan(ERA, 2025.5, color=POST_BG, **kw)
for gy in (1997, 2013):
    ax.axvspan(gy, gy + 1, color='#e8a0a0', alpha=0.55, zorder=1, lw=0, antialiased=False)
gw_raw['yr_dec'] = gw_raw['Date'].dt.year + (gw_raw['Date'].dt.dayofyear - 1) / 365.25
ax.plot(gw_raw['yr_dec'], gw_raw['gwt'], color=GREEN, lw=0.7, alpha=0.7, zorder=2)
ax.plot(dry.index, dry.values, '-s', color=RED, ms=4.5, lw=1.6, zorder=4)
ax.plot(mon.index, mon.values, '-o', color=BLUE, ms=4.5, lw=1.6, zorder=4)


def draw(sp, sq, c):
    xp, xq = np.array([1988, 2012]), np.array([2014, 2024])
    ax.plot(xp, sp[0] * xp + sp[1], '--', color=c, lw=1.5, alpha=0.9, zorder=3)
    ax.plot(xq, sq[0] * xq + sq[1], '--', color=c, lw=1.5, alpha=0.9, zorder=3)
    chk.seg(ax, xp[0], xp[1], sp[0] * xp[0] + sp[1], sp[0] * xp[1] + sp[1])
    chk.seg(ax, xq[0], xq[1], sq[0] * xq[0] + sq[1], sq[0] * xq[1] + sq[1])


draw(d_pre, d_post, RED); draw(m_pre, m_post, BLUE)
ax.axvline(ERA, color='black', ls='--', lw=1.2, zorder=5)
ax.set_ylim(13.4, -1.35)
ax.set_xlim(1988, 2025.5)
chk.label('a:ylabel', ax.set_ylabel('GWn depth (m bgl)'), ax, inside=False)
ax.xaxis.set_major_locator(plt.MultipleLocator(4))

chk.label('a:t1', ax.text(2002.0, 7.60, f'pre-2013 dry:\n{d_pre[2]:+.2f} m/dec (p=0.73)',
                          color=RED, fontsize=FS['annot'], ha='center', va='center',
                          style='italic', linespacing=1.2), ax)
ax.annotate('', xy=(2006.4, 3.95), xytext=(2002.6, 6.50),
            arrowprops=dict(arrowstyle='->', color=RED, lw=0.9, alpha=0.8))
chk.label('a:t2', ax.text(2025.2, 7.60, f'post-2013 dry:\n{d_post[2]:+.2f} m/dec (p<0.001)',
                          color=RED, fontsize=FS['annot'], ha='right', va='center',
                          fontweight='bold', linespacing=1.2), ax)
ax.annotate('', xy=(2021.2, 5.20), xytext=(2022.6, 6.50),
            arrowprops=dict(arrowstyle='->', color=RED, lw=0.9))
chk.label('a:letter', ax.text(0.990, 0.975, 'a', transform=ax.transAxes, fontsize=FS['panel'],
                              fontweight='bold', va='top', ha='right'), ax)
chk.label('a:pre', ax.text(1989, -0.60, 'Pre-2013', color=BLUE, fontweight='bold',
                           fontsize=FS['small'], va='center'), ax)
chk.label('a:post', ax.text(2017.3, -0.60, 'Post-2013', color=RED, fontweight='bold',
                            fontsize=FS['small'], va='center'), ax)
chk.label('a:g97', ax.text(1997.5, 5.80, '1997 data gap', color=RED, fontweight='bold',
                           ha='center', va='center', fontsize=FS['annot']), ax)
chk.label('a:g13', ax.text(2016.5, 11.00, '2013 data gap', color=RED, fontweight='bold',
                           ha='center', va='center', fontsize=FS['annot']), ax)
leg = [Line2D([], [], color=GREEN, lw=1.6, alpha=0.8, label='Weekly readings'),
       Line2D([], [], color=RED, marker='s', ms=5.5, label='Dry season (Nov\u2013Apr)'),
       Line2D([], [], color=BLUE, marker='o', ms=5.5, label='Monsoon (May\u2013Oct)'),
       Line2D([], [], color='gray', ls='--', lw=1.5, label='Period trends')]
lega = ax.legend(handles=leg, loc='lower left', ncol=2, fontsize=FS['legend'],
                 framealpha=0.95, borderaxespad=0.7, handlelength=2.4)
chk.legend(lega, 'a:legend')
chk.path(ax, gw_raw['yr_dec'].values, gw_raw['gwt'].values, n=2)
chk.path(ax, dry.index.values.astype(float), dry.values)
chk.path(ax, mon.index.values.astype(float), mon.values)
chk.seg(ax, ERA, ERA, -1.35, 11.4)

# ------------------------------------------------------------------ (b)
ax2 = fig.add_subplot(gs[1])
coup['era'] = np.where(coup.index <= 2011, 0, np.where(coup.index <= 2016, 1, 2))
for k, (lab, col, mk, sz) in enumerate(ERA_STYLE):                       # 243, 245
    s = coup[coup.era == k]
    ax2.scatter(s['wl'], s['gwt'], c=col, marker=mk, s=sz, ec='black', lw=0.5,
                zorder=3, label=lab)
x0, y0 = coup.loc[2012, 'wl'], coup.loc[2012, 'gwt']
x1, y1 = coup.loc[2024, 'wl'], coup.loc[2024, 'gwt']
ax2.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle='-|>', mutation_scale=18,
                              color='gray', lw=1.6, alpha=0.7, zorder=2,
                              connectionstyle='arc3,rad=-0.18'))
chk.label('b:y2012', ax2.annotate('2012', (x0, y0), textcoords='offset points',
                                  xytext=(10, 9), fontsize=FS['annot'],
                                  fontweight='bold'), ax2)
chk.label('b:y2024', ax2.annotate('2024', (x1, y1), textcoords='offset points',
                                  xytext=(6, 20), fontsize=FS['annot'],
                                  fontweight='bold'), ax2)
ax2.margins(0.09)
ax2.invert_xaxis(); ax2.invert_yaxis()
chk.label('b:xlabel', ax2.set_xlabel('WLd dry-season water level (m PWD)'), ax2, inside=False)
chk.label('b:ylabel', ax2.set_ylabel('GWn dry-season depth (m bgl)'), ax2, inside=False)
chk.label('b:letter', ax2.text(0.010, 0.975, 'b', transform=ax2.transAxes,
                               fontsize=FS['panel'], fontweight='bold',
                               va='top', ha='left'), ax2)
chk.label('b:stat', ax2.text(0.988, 0.975, f'r = {r_c:.2f},  $p = {p_c / 10**np.floor(np.log10(p_c)):.1f} \\times 10^{{{int(np.floor(np.log10(p_c)))}}}$',
                             transform=ax2.transAxes, fontsize=FS['annot'], va='top',
                             ha='right', bbox=dict(boxstyle='round', fc='white',
                                                   ec='gray', alpha=0.0)), ax2)
legb = ax2.legend(loc='lower left', fontsize=FS['legend'], framealpha=0.95,
                  borderaxespad=0.7, handletextpad=0.5)
chk.legend(legb, 'b:legend')
chk.pts(ax2, coup['wl'].values, coup['gwt'].values)

fig.align_ylabels([ax, ax2])

issues = chk.run()
if issues:
    print(f'\n{len(issues)} layout issue(s) above — figure not saved.')
else:
    _png_gwt = _io.BytesIO()
    fig.savefig(_png_gwt, format='png', dpi=300, bbox_inches='tight')
    with open('cl_fig_gwt_v3.png', 'wb') as _fh:
        _fh.write(_png_gwt.getvalue())
    fig.savefig('cl_fig_gwt_v3.pdf', bbox_inches='tight')
    print('saved cl_fig_gwt_v3.png and cl_fig_gwt_v3.pdf')
plt.show()
plt.close(fig)


## Figure 9: rainfall eliminated

Canvas narrowed and heightened so three panels fit legibly, the panel (b)
note rewrapped to clear the panel letter, and the local collision checker
extended with the label-against-label test it was missing.


In [ ]:
# @title Figure 9 - rainfall analysis
"""Figure 9 v10 - enlarged type (Jim comment 261) with every label placed in
verified empty space. A collision checker runs after rendering and fails the
build if any label overlaps a data point, a plotted line, a reference line,
a bar, the legend, or the axes frame.
"""
import gzip, base64, io, sys
import numpy as np, pandas as pd
import scipy.stats as ss
import matplotlib.pyplot as plt, matplotlib as mpl
from matplotlib.transforms import Bbox

# ---------------------------------------------------------------- type scale
# ---------------------------------------------------------------- chat 31
# FIG_SCALE rescales the drawing canvas while the font sizes below are left as
# authored. AGU prints a two-column figure at 170 mm (6.693 in), so text set at
# F pt on a canvas W inches wide reaches the page at F * 6.693 / W pt. Scaling
# the canvas down is therefore the same as enlarging the type on the printed
# page, and it preserves every layout decision in relative terms.
FIG_SCALE_W, FIG_SCALE_H = 0.840, 1.216
# ---------------------------------------------------------------------------
# the 1 x 3 strip was too short to hold legible type; the height scale
# is larger than the width scale so the panels gain vertical room.
FIG_W, FIG_H = 13.2 * FIG_SCALE_W, 4.20 * FIG_SCALE_H
_FS = apply(11.0, {'axes.spines.top': False, 'axes.spines.right': False})
FS_BASE, FS_PANEL = _FS['base'], _FS['panel']
FS_LEGEND, FS_ANNOT, FS_SMALL = _FS['legend'], _FS['annot'], _FS['small']

# ---------------------------------------------------------------- data
_BLOB = ("H4sIAIaOWmoC/y2SS4ocMBBD93MWY+rn+pymaUgCge5ZDIQwt4+qnO1DliXL3z+fX+v5+fnn+"
         "Xp8PX9//nq+Xo/3e/34+n78fYnr4/3BlbmsTDev2lIVjWqZiG8C0vACKlpGVaM6VN6Il6n5LiBL"
         "G5UsVeKL9IxKl5nHlsW06TA3syUVBLNmznP0LA32Hc0kS5s5dOXbmqnHaRZLrCXNuC7LJWS8dXSq"
         "41dLKeT6sRz+ECJa4nzgV7sCiYEYiTV2zrUno5kgMudlZOHNtHXnXsEp0swQ+RhY7SzXRt1CeJ+W"
         "4XUaOYJ4/g8nPm6xNM1uCaHLcqngOeWWOGNXCzvkLaFFB4xpqQtdZkatY7SwKdY6ztHJksAdp8tO"
         "OEaE1JpZU3NQL6G+Hcj9eh0gtlFZDMEMB9+hnSi0q3Pgh/i1EpmejBFOKlBujhoV4gfRTgFTqY4l"
         "iI8RR0Z6+qQgvZw+GdspegLBR2KgANKaFKJ4izhYBQg3NcIAPIXwvyx7T8EATI7XBkqLj3/Dhkem"
         "BQMAAA==")
df = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(_BLOB))))
df = df.sort_values('year').reset_index(drop=True)

CUT, BL = 2012, 2011.5
Cc = {'pre': '#2c7fb8', 'min': '#d7301f'}
Ll = {'pre': 'Pre (≤2011)', 'min': 'Mining (≥2012)'}
eo = lambda y: 'pre' if y < CUT else 'min'

m = df.dropna(subset=['dry_wl263_m']).set_index('year').copy()
m = m.rename(columns={'annual_rainfall_mm': 'rf', 'dry_wl263_m': 'wl'})
pre = m[m.index < CUT]
sl, ic, r, p, _ = ss.linregress(pre.rf, pre.wl)
m['bel'] = m.wl - (ic + sl * m.rf)
pm = m.loc[m.index >= CUT, 'bel'].mean()

ann = df.set_index('year')['annual_rainfall_mm']
prr, mnr = ann[ann.index < CUT], ann[ann.index >= CUT]
s_pre, i_pre, *_ = ss.linregress(prr.index, prr.values)
s_mn, i_mn, *_ = ss.linregress(mnr.index, mnr.values)

mn = m[m.index >= CUT]
LO = max(pre.rf.min(), mn.rf.min()); HI = min(pre.rf.max(), mn.rf.max())
opre = pre[pre.rf.between(LO, HI)].wl.mean()
omin = mn[mn.rf.between(LO, HI)].wl.mean()
pre_wl_min = pre.wl.min()

print(f"pre-mining reg: WLd={sl:.5f}*RF+{ic:.3f} (r={r:.2f}, p={p:.3f})")
print(f"overlap {LO:.0f}-{HI:.0f} mm | pre {opre:.2f} vs mining {omin:.2f} -> offset {omin-opre:.2f} m")
print(f"mining-era mean below expectation: {pm:.2f} m")

# ---------------------------------------------------------------- figure
WB = dict(boxstyle='square,pad=0.18', fc='white', ec='none', alpha=0.0)
TRACK = []          # (label, text artist, axes, must_be_inside_axes)
OBST = {}           # axes -> list of (x, y) data-coord obstacle points
BARS = {}           # axes -> list of data-coord bboxes


def obst(ax, xs, ys):
    OBST.setdefault(ax, []).extend(zip(np.atleast_1d(xs), np.atleast_1d(ys)))


def seg(ax, x0, x1, y0, y1, n=400):
    obst(ax, np.linspace(x0, x1, n), np.linspace(y0, y1, n))


def tag(name, artist, ax, inside=True):
    TRACK.append((name, artist, ax, inside))
    return artist


fig, ax = plt.subplots(1, 3, figsize=(FIG_W, FIG_H), gridspec_kw={'width_ratios': [1.0, 1.25, 1.0]})

# ---------------------------------------------------------------- (a) rainfall
a = ax[0]
a.bar(ann.index, ann.values, color=[Cc[eo(y)] for y in ann.index], width=0.8)
a.plot([1988, 2011], i_pre + s_pre * np.array([1988, 2011]), color='#08306b', lw=2.4, zorder=4)
a.plot([CUT, 2025], i_mn + s_mn * np.array([CUT, 2025]), color='black', lw=2.4, zorder=4)
a.axvline(BL, color='0.45', ls='--', lw=1.4, zorder=5)
a.set_ylabel('Annual rainfall (mm)'); a.set_xlabel('Year')
a.set_ylim(0, 6100); a.set_xlim(1985.0, 2028.0)
a.set_xticks([1990, 2000, 2010, 2020])
tag('a:2012', a.text(BL + 0.9, 5300, '2012', fontsize=FS_SMALL, color='0.45',
                     weight='bold', va='top', ha='left'), a)
tag('a:panel', a.text(0.960, 0.960, '(a)', transform=a.transAxes, fontsize=FS_PANEL,
                      weight='bold', va='top', ha='right'), a)
BARS[a] = [(y - 0.4, 0.0, y + 0.4, v) for y, v in ann.items()]
seg(a, 1988, 2011, i_pre + s_pre * 1988, i_pre + s_pre * 2011)
seg(a, CUT, 2025, i_mn + s_mn * CUT, i_mn + s_mn * 2025)
seg(a, BL, BL, 0, 6100)

# ---------------------------------------------------------------- (b) WLd vs rainfall
b = ax[1]
for e in ['pre', 'min']:
    s = m[[eo(y) == e for y in m.index]]
    b.scatter(s.rf, s.wl, c=Cc[e], s=60, edgecolor='white', lw=0.8, label=Ll[e], zorder=3)
b.set_xlim(1650, 5200); b.set_ylim(5.0, 11.75)
b.hlines(opre, LO, HI, color='#2c7fb8', lw=1.8, ls='--', alpha=0.8, zorder=4)
b.hlines(omin, LO, HI, color='#d7301f', lw=1.8, ls='--', alpha=0.8, zorder=4)
ARR_X = 2800
b.annotate('', xy=(ARR_X, omin), xytext=(ARR_X, opre),
           arrowprops=dict(arrowstyle='<->', color='0.2', lw=2.0), zorder=5)
b.axhline(pre_wl_min, color='#2c7fb8', ls=':', lw=1.6,
          label='Pre minimum: 9.3 m')
# Use equal-size emphasis rings. The selected size clears the blue point
# beside 2017 while keeping both highlighted observations visually consistent.
ring_size = 110
for y in [2017, 2020]:
    b.scatter([ann[y]], [m.loc[y, 'wl']], s=ring_size, facecolors='none',
              edgecolors='black', lw=1.8, zorder=6)
tag('b:2017', b.text(3300, 8.68, '2017: 4,269 mm',
           fontsize=FS_SMALL, ha='left', va='center',
           bbox=dict(fc='white', ec='none', alpha=1.0, pad=0.8)), b)
# Short connector from the 2017 label to its emphasis ring. It begins above
# the label box so neither the label nor nearby observations are crossed.
b.annotate('', xy=(ann[2017], m.loc[2017, 'wl']),
           xytext=(ann[2017], 8.88),
           arrowprops=dict(arrowstyle='->', color='0.25', lw=1.0,
                           shrinkA=0, shrinkB=6), zorder=7)
tag('b:2020', b.text(3300, 7.25, '2020: 3,997 mm',
           fontsize=FS_SMALL, ha='left', va='center',
           bbox=dict(fc='white', ec='none', alpha=1.0, pad=0.8)), b)
# Match the 2017 treatment with a short connector from the 2020 label
# to its equal-size emphasis ring.
b.annotate('', xy=(ann[2020], m.loc[2020, 'wl']),
           xytext=(ann[2020], 7.45),
           arrowprops=dict(arrowstyle='->', color='0.25', lw=1.0,
                           shrinkA=0, shrinkB=6), zorder=7)
tag('b:offset', b.text(0.025, 0.955, 'Mining offset\n−1.8 m',
                       transform=b.transAxes,
                       fontsize=FS_ANNOT, va='top', ha='left', bbox=WB), b)
tag('b:panel', b.text(0.960, 0.960, '(b)', transform=b.transAxes, fontsize=FS_PANEL,
                      weight='bold', va='top', ha='right'), b)
b.set_xlabel('Annual rainfall (mm)'); b.set_ylabel('Dry-season WLd (m PWD)')
leg = b.legend(fontsize=FS_LEGEND, loc='lower left', framealpha=1.0,
               ncol=1, borderpad=0.35, handletextpad=0.35, handlelength=1.3,
               columnspacing=0.8, borderaxespad=0.0, mode='expand',
               bbox_to_anchor=(0.025, 0.025, 0.95, 0.25))

obst(b, m.rf.values, m.wl.values)
seg(b, LO, HI, opre, opre); seg(b, LO, HI, omin, omin)
seg(b, 1650, 5200, pre_wl_min, pre_wl_min)
seg(b, ARR_X, ARR_X, omin, opre)

# ---------------------------------------------------------------- (c) residual
c = ax[2]
c.plot(m.index, m['bel'], '-o', color='#d7301f', ms=6, lw=1.8, zorder=3)
c.axhline(0, color='0.4', lw=1.2)
c.hlines(pm, CUT, 2025, color='#d7301f', ls='--', lw=2.0)
c.axvline(BL, color='0.45', ls='--', lw=1.4, zorder=5)
c.set_xlim(1985.0, 2028.0); c.set_ylim(-4.35, 1.35)
c.set_xticks([1990, 2000, 2010, 2020])
tag('c:zero', c.text(1985.8, -4.12, '0 = rainfall-\nexpected\nlevel', fontsize=FS_ANNOT,
                     color='0.4', ha='left', va='bottom', bbox=WB), c)
tag('c:2012', c.text(BL - 0.8, 1.28, '2012', fontsize=FS_SMALL, color='0.45',
                     weight='bold', va='top', ha='right'), c)
# Keep the mean label inside panel (c); the previous outboard placement
# forced bbox_inches='tight' to retain a large blank right margin.
tag('c:mean', c.text(1986.0, pm, 'post-2012\nmean\n\u22121.72 m', va='center', ha='left',
                     fontsize=FS_SMALL, color='#d7301f',
                     bbox=dict(fc='white', ec='none', alpha=0.92, pad=0.8)), c)
tag('c:panel', c.text(0.960, 0.960, '(c)', transform=c.transAxes, fontsize=FS_PANEL,
                      weight='bold', va='top', ha='right'), c)
c.set_xlabel('Year'); c.set_ylabel('WLd relative to rainfall\nexpectation (m)')

yrs = m.index.values.astype(float); bel = m['bel'].values
for i in range(len(yrs) - 1):
    seg(c, yrs[i], yrs[i + 1], bel[i], bel[i + 1], n=60)
seg(c, 1985.0, 2028.0, 0, 0)
seg(c, CUT, 2025, pm, pm)
seg(c, BL, BL, -4.35, 1.35)

fig.tight_layout(w_pad=2.0)
fig.subplots_adjust(right=0.985, wspace=0.40)

# ---------------------------------------------------------------- collision check
fig.canvas.draw()
rend = fig.canvas.get_renderer()
PAD = 3.0
fails = []

for name, art, axx, inside in TRACK:
    bb = art.get_window_extent(rend).expanded(1.0, 1.0)
    bb = Bbox.from_extents(bb.x0 - PAD, bb.y0 - PAD, bb.x1 + PAD, bb.y1 + PAD)
    pts = OBST.get(axx, [])
    if pts:
        disp = axx.transData.transform(np.array(pts))
        hit = [(px, py) for px, py in disp if bb.x0 <= px <= bb.x1 and bb.y0 <= py <= bb.y1]
        if hit:
            inv = axx.transData.inverted().transform(np.array(hit))
            fails.append(f"{name}: overlaps {len(hit)} plotted point(s), e.g. data {inv[0].round(2)}")
    for x0, y0, x1, y1 in BARS.get(axx, []):
        (dx0, dy0), (dx1, dy1) = axx.transData.transform([(x0, y0), (x1, y1)])
        if bb.overlaps(Bbox.from_extents(dx0, dy0, dx1, dy1)):
            fails.append(f"{name}: overlaps bar at x={x0 + 0.4:.0f}")
            break
    if inside:
        ab = axx.get_window_extent(rend)
        if not (ab.x0 <= bb.x0 and bb.x1 <= ab.x1 and ab.y0 <= bb.y0 and bb.y1 <= ab.y1):
            fails.append(f"{name}: extends outside the axes frame")

for _i in range(len(TRACK)):
    for _j in range(_i + 1, len(TRACK)):
        _n1, _a1, _x1, _ = TRACK[_i]
        _n2, _a2, _x2, _ = TRACK[_j]
        if _a1.get_window_extent(rend).overlaps(_a2.get_window_extent(rend)):
            fails.append(f"{_n1} overlaps {_n2}")

lb_raw = leg.get_window_extent(rend)
ab = b.get_window_extent(rend)
if not (ab.x0 <= lb_raw.x0 and lb_raw.x1 <= ab.x1 and ab.y0 <= lb_raw.y0 and lb_raw.y1 <= ab.y1):
    fails.append('legend: extends outside panel (b)')
lb = Bbox.from_extents(lb_raw.x0 - PAD, lb_raw.y0 - PAD,
                       lb_raw.x1 + PAD, lb_raw.y1 + PAD)
disp = b.transData.transform(np.column_stack([m.rf.values, m.wl.values]))
lhit = [(px, py) for px, py in disp if lb.x0 <= px <= lb.x1 and lb.y0 <= py <= lb.y1]
if lhit:
    inv = b.transData.inverted().transform(np.array(lhit))
    fails.append(f"legend: overlaps {len(lhit)} point(s), e.g. data {inv[0].round(2)}")
for name, art, axx, _ in TRACK:
    if axx is not b:
        continue
    tb = art.get_window_extent(rend)
    if lb.overlaps(tb):
        fails.append(f"legend: overlaps label {name}")

# ---- pixel-level check: is anything actually drawn beneath each label? ----
import io as _io
from PIL import Image
boxes = {}
for name, art, axx, _ in TRACK:
    bb = art.get_window_extent(rend)
    boxes[name] = (bb.x0 - 2, bb.y0 - 2, bb.x1 + 2, bb.y1 + 2)
lbb = leg.get_window_extent(rend)
boxes['legend'] = (lbb.x0 - 2, lbb.y0 - 2, lbb.x1 + 2, lbb.y1 + 2)

vis = [(a_, a_.get_visible()) for _, a_, _, _ in TRACK]
for _, a_, _, _ in TRACK:
    a_.set_visible(False)
leg.set_visible(False)
buf = _io.BytesIO()
fig.savefig(buf, format='png', dpi=fig.dpi, facecolor='white')
for a_, v in vis:
    a_.set_visible(v)
leg.set_visible(True)
buf.seek(0)
under = Image.open(buf).convert('RGB')
W, H = under.size
px = under.load()
for name, (x0, y0, x1, y1) in boxes.items():
    if name == 'legend':
        continue  # geometric checks already confirm the opaque legend is clear
    ix0, iy0 = max(0, int(x0)), max(0, int(H - y1))
    ix1, iy1 = min(W, int(x1) + 1), min(H, int(H - y0) + 1)
    dirty = 0
    for yy in range(iy0, iy1):
        for xx in range(ix0, ix1):
            r_, g_, b_ = px[xx, yy]
            if r_ < 245 or g_ < 245 or b_ < 245:
                dirty += 1
    if dirty > 0:
        fails.append(f"{name}: {dirty} non-blank pixels drawn beneath the label")

if fails:
    print("\nCOLLISIONS:")
    for f in fails:
        print("  -", f)
    print(f'\n{len(fails)} layout issue(s) above — figure not saved.')
else:
    print("\nno collisions: every label clears all points, lines, bars, legend and axes")
    fig.savefig('someshwari_fig9_rainfall_v10.png', bbox_inches='tight', dpi=300)
    fig.savefig('someshwari_fig9_rainfall_v10.pdf', bbox_inches='tight')
    print('saved someshwari_fig9_rainfall_v10.png and someshwari_fig9_rainfall_v10.pdf')
plt.show()
plt.close(fig)


## Figure 10: discharge eliminated

Dashed handles in the panel (b) legend, and the panel (c) bar labels and
panel (d) notes spaced apart for the larger type.


In [ ]:
# @title Figure 10 - discharge analysis
"""Figure 10 v4 - discharge eliminated, rating-curve decomposition.

Changes from fig10_discharge, both from Jim's review:
  262  the two pre-mining curves in (b) are dashed, but the legend handles were
       too short to show the pattern. Handles lengthened so the dashes read.
  261  type scale enlarged and made consistent with the other figures.

Data, fits and panel content are unchanged.
"""
import gzip, base64, io, sys
import pandas as pd, numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# ---------------------------------------------------------------- chat 31
# FIG_SCALE rescales the drawing canvas while the font sizes below are left as
# authored. AGU prints a two-column figure at 170 mm (6.693 in), so text set at
# F pt on a canvas W inches wide reaches the page at F * 6.693 / W pt. Scaling
# the canvas down is therefore the same as enlarging the type on the printed
# page, and it preserves every layout decision in relative terms.
FIG_SCALE_W, FIG_SCALE_H = 1.146, 1.146
# ---------------------------------------------------------------------------
_STYLE_W, _STYLE_H = 11.0, 8.0
FIG_W, FIG_H = _STYLE_W * FIG_SCALE_W, _STYLE_H * FIG_SCALE_H
FS = apply(_STYLE_W)
HL = 2.4          # legend handle length, long enough to show a 6-2 dash pattern


def mk_p(x):
    n = len(x)
    s = sum(np.sign(x[j] - x[i]) for i in range(n - 1) for j in range(i + 1, n))
    var_s = n * (n - 1) * (2 * n + 5) / 18
    z = (s - 1) / np.sqrt(var_s) if s > 0 else (s + 1) / np.sqrt(var_s) if s < 0 else 0
    return 2 * (1 - stats.norm.cdf(abs(z)))


PAYLOAD = "H4sIADD1WWoC/319u7LtuK5d7m9ZPYsE37lDJ44c3yrf4CZ24KP/NwZAiQBFna6d9BpTEkWCeHPov//Hv/7z73/+/a//8ff//vUf//qv//t//lsco/4T6j/U/lrrv/wX049q55/874m1fyj9pUK/1P5i/JHFBsN/OcVfEag8WMN1Mf5RTfUXEu46RjYoXzn+qORfjn8x/HqksP6L7of8+Fjz+FUeHNnB8RMCBhAz8Q1wmxKzAyn+9fYbHVgqBor/hPzX068VQDwOh8X+V9svJ2Cx81/Xf+Z39E+gv1J+QW5BDok8nfUX5e7JQyn85cKPlKueN+3/BH5u+svxx/MEyCG8PDyRRUdUDMSPCn+JfnK79faCUP7jpUlDoNTNBNubJ7lDxPzgdw7hO+T8CzJ/lO1VmYXmL7VfFyzWflo++R2/I9+96SjcPQr/wz2GPnmEj3uUfyJk5Tfo9foVa8Dv9uty/2qgjkWgVrNcxzOU+kKxROWPhSrKdXZYDNHAEjUVm+eqIUtEf5R/fGdAzUFU/0j+6NZ8yErIDqjVLeyQGU5/fQJhGCRjbkb6lezfWSAKGEQSaaBUDVYgu6Xi+dgODquQed6mcsceyEEQVR5ilnduLTuQ34w36K9F2YDkr8Rk5cbXyL3jURCGqIP0V0KUlYJUkgNFwIO8U+Q91057bsiihr8yGtQQ/7DS+PwhpJ/vGEQ1pUAf4xoiQQUbGL/L5LH0N35FnkWxeoilq3TRSuE3HMQiH8ctky29H42/TGkqvBZ2+W+IFUSBsDx7UgFWOv0vDVU6oVcH8Tuk/KuNxzzs3RIGy/pjVCDdIdieLJoRyFx2hTLEhSGe6YFtbhEWMta5TcdgIZa/CvkrU+l8vXqBGuysW/DuLBOFDMjSmP8ab/gqa5LdE8RUlcKaxy2Ygk32Z+fxikoZ3YMsx7F3TBB0RfkaHDQuCzzvsAqBz27GRAKpE3Q39tHIDuT9nHOQbctyXOyVA/u2RDYsVYX8bVjkhzBqfHmtU+G1Uh0YsUfLfAs2CRZLGFSb2rYEe2HErLLA5qra1j69u9/xirNZo6HLayASgZ0aJGSH8NyyOs3JyV6EJLNgs37IWSTMIZgsdg9E9rJBCMaFNXBKTvQE4TlkG8kLO35L+qPIuE4rgJgdwtc0qPPxa/aSrNu7yVOSA2A6xGXYxlwgGpE3plqA5oZdxKbwsGXUHuHZZEsUdGtaqGJOWZzU22i1nYQiTi0ca4KRl9dcUBMLHvINHYzx/UNWijEnjIN/GNwYG4SnwxHokPqWv+7SRfWlAHUCvRjensX9Q+yRRqwHZDeOrzvCqPF895+YBqrZYSzvKQ7x8DA4Ot8Eu4b3Pn6iWz87jNVSbKJkxb51B7IuitR+QXZGqV9PEAetsZGcPgM5jNUSr47qcqoWI3jCrH6j+poO4f/lKc+q559RkewbvLfYpccBuCEoesLGWU6fQgS3gdj+CxSag+BFkVhOu3tJ9g7rS1YNqtEdAgdpABlY74VktZtx3i/17kCZb3X7WCKjw3i6KyVW3bKgjQxYoKRioOn31FodCFsktwOWLVYxx7EW6Bmo5ZZO+4imcaHMwiZ74Ut85Ze8OJmGuGcQyxtLMmOQWIjC+A1yCC8jT7Pqu2IvylAe2DfqM6fssFjMclM4yaD8jjUJz8F08NyjC+STHYCmPmNyEMRT9B8Wy11WoT56mQtZcnYY3CgODKm85qBiITkMFB+aXYiWP8bcIMqI9UZTPZAdKMJcRO3we50VYBK1wzPExq2qj5i7AyHcPUi0AxVoHyEuHuUuPio/P309YojYVtk4ECL3iCFqovCLqo+cHjCLPEhgoPO7NpdAshewJa1pFETuiD9CVAySsZI1zmDCKETBdCnVw+rHd8kiDJE9ZdllVkkIJKtJ6kTWYF+kYq0i7LXuMXKYSCeLiqYCupuCJgY45DCtS24ORDKgs9IaUZIBy3YLiohmYNurKPWjKGURAhZx3rYanHU30xLx3RKCwXcH8gJwuCDaOxrVmGXdoRM4GpRdVWN1IEERx+k4uXWHYSH2YfiRetdEDoTJZocgqzdmLxS3CK6aPpJadSB7RlB0unbxY42RgWh/POGVvFshEBz7jh07fs81RexKhhKK8G2WkRQEbn1WcVwWsohVkZs16LQ2HAK1VeA+iSO1kDSVXZFrHMCX8H5vMCktdAexctOEwDaCLPMc58qa7S0YfM96x1HJXVjEN8rPdjn63PI7VnSDg4OmYUVIJ/e4yAZh72fk6bCVcxQhP4T3A1kNEjebYKDIbuEAL8sCuc1SZqhMqUXJvOB68ij7kpgl3NXeE0kfjtiqCCPcs+xARGZlpupKstgQw8ExhSrPQA5DaB6QW4GAf70udgKvN7vzmkKwk4uNwFLCYpK7qm6DiZ8TYVuyTT3cIGucwUpDN3Q1kMQIrc2cWCkOYnXC3rbeMT8jqSL9SRwuiQ+XqRMITv+Yup/9mZNbIL+DImkzPkr2FqR6ZDp6wyGsRBiZKaBuoISLxnQKqDlEsxwzSWmQLJN9p6jWy1eJUIZkEYfNACmGfwjt9O27HWATtTD4rwJ9vj0ysmxWEOTAgPTqMCgQxM1RRcBe2FWH0BTO7yd0iaLYP5SpMjmJOs18YTuhk+/eoKvyIHm7+OR0FRyqp/tPnYzWHcZ7MMF1nOGHGQ3EmmVJ0oAy6uow5EfLzD0bX7WKWDdJImpeujuIFUOeM22WLorznu7lSd1B/OKAkhetdot0mE+i9ah2y3SXRMqTbb6hhDzGDD3IICTqZ9xilx1ETTKxbb8o3Rf5QEYQkn1Vug9/2i3GSNvLbrTDK9j5iLNVlMJR7TQRaYm5ddFDyA7jSctzQsnqNSlPiCZoM2iu1YNI5nbW8gUWnHfYUVr7nMY6h7lSeJqZJ4QImgVcqRhNzUuuWOsDa8yajUfMPLNLITkIbtutoOKSYU3Hd1ymkxxbOVkvTcfLoII6hXZQMFlYW832p5YdpinnTiqZFoOi4lumGSYFqg5kAczIharzZMeiYQHqQ+o6OYiHWXnNNHVfk7+O5IFNcrscjqejUe8zZGBwZmt6WdisAFXNFlS7PFoAYk2l4UIxy40tOlBS0kg5Zwchlx+m03ZKhPPvpFAAJczWKZkawg0hQ5jukDobCLE4Emzq4jggig84/UgyUJqaoWybUiAohjgH4aAsJbU6E9vUPAa5LHMU2Q5QxSdMfZIjOQzRWdZsdh4WqloHhO8ILDkI+cE7XVuCvQx7FDW6GDWV6iCo1yrCIaplQdgAUOZN3fxzlKFVhs6/i7KdYGzIgTzJHe8vbl1zbyqRBPKD08SG+vEIMToROXkd5vL/xx1VtDQT4ME+/44qnuxwP2Y6hogxdlaHl++8pzGTvCj4FO9ADDE1QSKTrKmC6jB4yQ17YlVUKUSpWCC5GWeuLz8bSjEUYvusCqRqMRHseteBk0NYRnmjeLsRw8y+qCuLTLZD4ILwKzeXNYthqsgWpi+57KRi8DPb1G89hYO2j2EKeURKqGig2B2I3HQKsza1fKMYZqESLn1yjotC7JnDMyJNTxRyIPI9TTQ2q8N6WmypgotfRmqwvnLA+kNCaiHMygs9UWPUygeHUuxDTQ+ueRB59FSnF1/Ivp84WJHDAxFoRt19h4ZovF91dNGAcuu/XlSqEJl3B/LKZOgO9fhiOU9AlGJaqjQd6ZocBg8PYao6xWbgKu7EnmEarp6iGIu7WChV9891ccp7HOlWeOQwSXzHu/NgGEySwRGhbDWF4huDhYeDrg6RgdQvYJlXZzMlh+Eynl1NHq7sXoy3Z8DvRzNrnR2ISziY0TJ4OmZNY7wFv4cntUIOhHcQbt1f3TtJdjG2OpVud5NY7wlWdbnKKVHLGuzaVd0XLBfZgwR57+KlQqLsbbvqTzVvcTmWE2M9NfpMNRUHQUpLvaX0yR/FOBtWIIi69VdmV0Gp+0Xxz7H7zAxoj0qq7LjpDNRT5U1/KMXYNL04C0XxlGO48+LRYbz81G+PwryRuivEb6QBQHYQvCie+t6dHxJpuihw6dSXeWJIxXix45hB7lpqmr0oedoUag5BGnnMHpOly7X2IJFNc+6LIhhh09s9rnWk2XzCax+0EFlORlZ/R2oBdULde0iik2qe71iiB+Emo61IN34Zp31BM+lZEDbLwpeQP4YiKc7CmnWIMC97D1Bryiiba/EieZDnriWatZRs53VIF064izB1pI9xSk9CK3f1p1P1IEbPyqVhS6zIQpQjbG0b2h8Uf81BKM2xJ5L+TeFMql8y1+r/b6usKZ44TUOwW4dmkieGMEtzMToM5mjM9Vv+RpKwl4OyOtXoGopApFXwYQtaQGg2SXRkY3pzyOxhkk6IpfrSzGvX3GZWIz+RkYLQyygYqsthnyYqqvcwA72STwUf/SGho6LMah8VexcpnPZZ1IsrVIlJVIa2MRVbXgciW5/DJ80NPw/L9+aGyzVWtBTzzCphsiUzSw4RtdOhF8eHwxvzzD1p+mfApbcIHhpousK09Hyejj5xxKWqo45TLlx/KIt0N+SY/Zyn4z9mdoJiPrl3efr9mV3OqJmg7u8BNVtQXpB0wsj0cZshpVvcpm7+bp751IgKi5ZJSv6YMZiDIBV/tTgrfxylRMBmrKNh0y9hlC4gzZSP5fPHcnsl8JqHMYmaxNdGFyz7E2rHcleoBjT2WIalTM3KWru4x5fZgJaKpttDsDcTV5Y0XHOeVrlzAbXrdak4SF4oz52Sy1HHlemfUmADFHWB0nFPlelxRvY4h/Zj9I/fxbvZSkoVFhHDysI2MGVrS9bZ74J+MpT7cjFI+kdTKwXIcuDqLFLg5WWan8RlvHOqHV1dfoPXmaSaTbtkuqQmNrTXaKs9RU3F8kLAHZZemaNHXWczE0JUxHzG7mjfL1yOqhPYjwmYeKdWKXWVxhaOO0azrOwH1jpT9jF2B0K7sbnvMj1nTXmnXPuAeA3jc98ZV56ohNGSRSTlyoYX9TYTSdZZWytavnLDkVRsHnB9xi86AIkHXcTn7+3ugCIoprGSfYrAwylQqMOYqTY7oBBKoj306UBSBFJeMNiOWHYh2gGVIZP9y8tssx2KEqQWcfLRs9HcLELOCsvaTZFYMbjtqLCInagGKiLkNA2siS7vnC4h6yQ74Cx4d06XYbhtw1SZ453TTWnM7Eexkybdf6UnSfcho1gdKM59n/7CKkAD1B2BEiKJ1jwqhDat9mMD0MLi7wK7Skm1Qs/23bXE1vKsV63mIQUlrS76YrhFVfcLaeo0m0bIgVJwkXhAMmtmNBKHSuAVtGqbHQYVyUJcsCOqfaBk/9lOh+E2vSBoPCi655f+6rMEkUVUhgkJ+ixAsDWSTj6HSNmLZQFu0Oq+UwS1MvTUcnQ3jtaxz4ow29gq+yAfpV0T3gUlL90u1Y6gSPsfwUhCUruD2CL2rq9aR/gYQ9WkxxT24pEibdHTkyByIFwAVWh4hn1y0/w/u8picshBKHDx07SJLRio30F+FPk9pv/1d4hTizSIsqAtZdSnhLZQZhTflgPV73Ql4gdpiD57YV3EleWbl0X0JmUHYVuS9FFgxdLHLTSxEsTbhwCQw1SRV9FJzSBSbYEJBVLi1/ik8tKkNNyNvte0PKFfpjUg1SH8vxWhj8zvArRNdaBiwouyclpjammcDRkCGUQKZVnaGXkZanaQ9hDyrmPoHEqOqZlHhXbsNvExpmKu0uPCzk04muVxd7GijCqzkD5/hk4NQrcAdu/yP8ct+jxfYrqSh6BpQ1D1navFJE0ZU1MFit5uB6IRbVaSpPxiMSkGVLWVvdp1kKxlypKgg2bKDiNN74nXFYOdLWnvl740SENxazEk+dyK6rTVTRo1Pa9lpYD92wc5DE5uRicvntcsFnXOAhJB3a+9pijRXQahP5cboyTg5TBV182RU3WYlLnEq+3LhyY9aZCRnEdfO7awRUhqXbJvng1BYSYqa5sb4plUhVip8txUkcBnTinM00TsyDXcj8JJtPRnENSse+rYw0ZhFoc6b1fxtXKvDoNAIEclUpYsJkFj09zsMEV7uo8IRGKPOupxlmd+FUWUQTgEJ9kOd9suTWxaqWY5O4a6+jtEihz3izY+p6cozDixs7/QRdlTdRjLWEWrdvRlOAUlZ9Lz7O4rGzik40kkfp1PIE21V4lMZWJWG5ViKN0M3pkQzuPJDf0ZolKevgDXlEb9/h3KiBnhYf+ZV5OMtUiPiml2EFRbU8GycjrjXXS8WvWsCPJj0mlh9DNJ+j5IQ1v4t7L4HAUTh/hcONGfoVFViod4Tv3+2UC7Q0qinY8P1fJBkbvJ+2SHiI+JvCw2V/cQImi4S311MCpCd/W1r1I66VkK+pNumvZbzsBExM9Kw5kvhaKEEnw7dOONj5e4D7aVgp/ZG0hWN0o811i8PaT9sAM2qneHiP/8g/nr0V+TAs4iDbxFd3Oi5eoCX6r9PBCTppowguoQBCBS2e+rF2siHSnnJM9J9iI5rCgVU4htLtlh0CVdMhtQv0fZiDM5IscQZAEfw6EYkoU4kiqy38PHPSRvjSY/cS2eJJ1C2ME8jAG7dHZb9XdQHx1aXdwo8hhOCdw1iOBWtepJiVkQoy/VFmdevEFLwF1M5CCsS55Rb+sbhqdL+I60S3UYFAobsQ5rnY9JG4qzRZhCxWIg818dhkQ7xS1HQHddKIeZPijNQTBW7HhDTZ9T/PqzhA63CE0Ei/T1Ozk6GPtQB66Hr92lvQRNKrX9I7uuv4PNYj0nL0wlft4PeRS0QzX5nRF8KV+L9CXxKzwU79p4N7WBCSG8CdOdrg5ihVS7IsGKWFRPWyJabJYPDakd0jgz9VIhUTzyLNknjOtjorW0ABMKd/rfPAe6hSdYnO6PxdB0UGowuGKeLBKlobrCoff1eX8DitIGh+fU+LE3ozaKYqL5Z2WcUiREs4ubdykrzGZy3QrBSyGY/8YreZQamt3e/LOEO+SzyqJ5EJXk5AcOLmYDSVc4xXmHvkEdL4tojbd/+3gL0gADVhvH7f0doKOm+k7DQtJZzlM0R949hEITos+2SpqKwGeXI/SMOOsdivsdTFpDCNJMAWBC0sUU5BbNQlmNg9ThumkvVQyRaVFx78FfhrRE1pccfrH971ADkq7lhoN5BipSAcoa8TVKHzMtGWacXUnfUk4z7/Ho5krVYajh9qhe7UrRKobTLfUnmvmYoCWarfM1k+ZhG2WHSYRSZ2ddc1MoXYQR0V+RXPdKCijKYlaDbEtUqPxteUUyzz59O6s0u/GpyVlU6ApymJz5x5myP3SLhs97QBlqtNyMd0azEEi5adC/WiuJZtN+DpLbH6bJYWLw92aGgVJ3mBRtigSMoX5tc03iw92K4m6Rw2DO9dCZU+1PZZnXA243+3ZfW3joWQXS0PKcS9DfSdY5Tmc8e6iJy6GO6PGIBt2VaBhvjS2Lg1jnZQ74xEi13j2GYk/XFGCI7fP2yBTmGUj0YGZq5lKL0A58Onh3oRvBkRiD7hCUTjNEv/2aA3j2G1pqvu2C/mxIekoUvr2zJGxLnKbAASQt6ANjzuc9n2aKFqU8aJB0Fu80jz3w+g3o6mQA0uPAiNF5ftrXY2h2Hosc0udohKqAveyG0aw1TjdVQVAfO1P5uIH0lFNQt4GOpwn1Z5JvEatP9PXSog9jxwbhnz3JfoXgNrCfh+noFhE3vIuPDIWfPkYgabJKGszRUjlp9qazkMlhOTR7VQ8idXI3PlD3V/LbryvDMthp1hJxlDppCOMgMfMZf/1UJ2me4kDGmzTnWT3W9RSdhh/1kMPS3/HIK0r6w5U7FRMeDjkT2VfHBKW75hjnhKVjzVF/J333UUPrsEJexeCwSgtDW+d2FRJ2CNI3Mw/WnDYe3IqoSXKY0O9ETfeF4K+D1Sqa5KLgL5NC58wCd7Od5sGRNvVB+dgnyubA4oec/ZfaSDNhnqavXbpHpFAvDq2PGexztECKtn7sOo+gyS6o81KqRwZUhYTk2zWapsDmfpY9z+Q7CmdvQDeg+C0fyinPTDxrjeC1Rp6JeCEFaKsdmPJ95r1g4do6K61IlNwFpiw+hRvK95l3Uo1K9ThneZ5+76pQY/wac5FyddVX6+XrZnIiPmXVux+aSttwJjFF8tmTPM/N0yiary727sIJVNNMQa6DxorhBDFmVQLpQR6DUU13v/GqD090WNT40Xmeq69QQzC04xyhaOsP4ShyvxtAz6m5PKsKAz2mEnmP7DAsRph9UE4EmmYOpViFPIgdZNfzW0Xrk/71ulbKSNzd4Z4m5QaUgKsoKfKXwR6HqI8zQVt+ShEVUVt/SnuKIGWRhccLDlz3GE5xSA8nikn+jlAxbWJPrYyksQhKvA6kuT49ojzbZGq8e9SSx3Dwl7oqcTut2rJNKGVUl8XVU8/CaqVRtEcQBGZ1sErxiHg+w7o3eVZMctWAOrePjaEVE75xzhJjOoRnFuwkRUy4RxCowGNsppCSZ0d4GZJNTc9jymwHL0Uj3mW37rYrdBzmb8dHz1jj5Igu85pPbc+SYyiiqMzf5RTE0Fg3nDMFZZ6IkKbSPzkwtZCsDQIa2of4NTBlppDCKPy7o+tbZod4lmZrTE12EPKSWY7xOgNaZu94ynlWK/xlsysiaPBmMTl01pCVkq7DTg5DG/IIGmqY5GOZkWQKVYP4cewl098J9V3Q6oXZq9qrpiedNQ1PHupCZoFJGCk7CLmOJufaMez8MY/a7hZI23iim0g5z4a6ltJbLaqOCVYhWJO0aq0Ogqlm6ZRCbl4JhDKP1KYqhhptQ/46mGv0T4vDOOyrKqkGOHvGPgtDW+CKbs3ePIbDLqj/SpRVHYSuInTt4465GWz20MP7r1L3/Jg9bSSDz6XhLn3/DhtV0p1WEZXZbdaqhlPZ7Bc9MRhD1Pp8OafXy+zVb109uO0OOFcWNZxIHkHw2TTtR5+3lmJOnvktyh4ZyGJ2ILE7RGZcHxrTxzafVET0+3Ja3kShBY1SjF0notA0N951YApNU3KvI1NoReuDQR1TKJIE3YGWHRQeDi3Us4PSz17pyEFpFl+vAzsoqcN0HchBSZ3b680HSj/7go4PlObBqetACEq/56qND5TUQF0HQlDSg6LXmxCU1ERdB0JQ0jbI68AB+oIWCSjNs0XXgQSUZjPcdSD+pJkSvQ7En6Qb4TpwfYLs1d7SkX3S5Be4jnSfUM4GdXSfpM1r14HukxcgPJdtfJ9Y0+KwRfiJbLu9zjJ+0jwtdb05P2ePy3Xg/EQ3/HDYIv3EIRJ7R0v6mWZV/3qTfqIvsjrIkH4mMMQ60JB+ltlAfJ1IPzlqIzsaR/TJYO4ONESfye70ndyTJz1VBxpCz/1KS+KZNJ6+3hyeIBYiDy0OTySwqgMNiyfNo5DXgbmTVBFfb+ZOUuf3elN3krbaXG/mzkgajF5v6k5A2UEPdycggxjqTiHpcNDD3Ul6Rus6UHdC3ZDDDF1nfnbzga4z++E7tk4Ig72rY+tMv6k+DmSdSSKi60TPWWYF6DrRc6aZmbtO9JxpdhZfJ3rONLuGrhMlJyxJd6Ch5KRHyA6cnAwWe1vLyXkr/zcL56ORXyyc9CNyyMPCCYl4nuRpOAFFBz08nJAWe5Uh4gSUHfQwccZ4m9adijNG5TW7XlScMf4GGWRxcQJxwMPFiSGQHZ5j42QrPzx4s3Huc2HYOMU0WczScdLsjbhOFJw0+ZWuEwknC3P0oCHehOWqDjTEm0jI2bl0ZJvQUeRAQ7BJs735OpBq8pCHxwypJno9zGAdkSbrkEgOM0SaZlvuNJpuF2zUmTy5tTpsUWc+zsyLOZM0r3cdmDMJ6z+hjTnT6MWNOJP0SOl1IM4k7di9DsSZ8bFnb+ZMI7eeORNUedVAjjqTZuHtOlFnptn4d524M0EMZkHHnckaPlQHLu5MmtyZ14k7M82c53Xky6zz5Op14shMv7WuniMTrZTNQQ9JZpQj7QuyJJl2ITaOTFqyt7Fi4oihHYZlxeSrlh3faDFJ007XgRUTMUB1mGHFTE8YdKDFrJMS8DpRYTrztFNhJg1LrxP7ZbI7aGe/TPMrCNeJ/dJpkJ3xMk3OiuvEeMngEpqN8dLuJc946QyKp7x0BmXjvCQ9eXK9KS9FuhdiWS4R7pDDFs0l/ewALcllmqcUryPJZfaP8xyXeZ5VvY4kl/U3qgMtx2V9Is4Dr2XWDs7rRGuZZk/ydaK1TDPvc514LdOk2r1OvJbJ6N2d1jJrj+R1YLWErFnMsVom47ZspJZJ+yOuN4+luNoOengsYeGft/NMlpMI8XozWUJgLGSoLCFl0UEPl6WzQ57MEpBDHjZLIKE77KGzlNDZDt4SWoLmymOG0DL/3D0tn2WaKbnrxGFZrK+081Yma3ReXJX4jUUdV2WavXjXkayy/kr14MNVmSff2HViq0y/nB22yCrhp1vMklWyiFF32CKrTD8zN56eMs2K1HXip4TYWtARVCKV5MFFUCknPBdkCSpJG0qvA0El/fJjdjaCyidR8SaofFI7b05KMtvVc1LaeHTjpKTfEhbPSWlD1Y2TkrROcR1IKcms50ZK6YzpRkrJeiHZ6ywr5Ur9vZkowdBXHWaYKFlJLZu3M1HCU+8OXOyTuHDDFvtkmnn060Q/yWOtFrT0k8kGDjv9ZHoSUG/6SQQ43WGLfhIvY27q6CfhCpLDHv5Jsjt9Y6C0UefGQGkzBxsDpQ21NgJK+i3zvBFQ2oh0Y6BE0iM7bFFQWr/Dc1DaVMnGQbkNw3JQ3h+3uQ4klDR7/q8D8SRmvzvMEE/yeicLOuLJNM94XkfiyfZbtt5zTZJWY68D1yT5iyzXpNUaG9ekXdKNaxI3zAazXJMweR57+CVpnsS7DvySNA9zXQd+SbL+684vmebpj+vEL5nnQcTrQDCZ5jdmrhPDZGEzXD1oGCbrL9srHamkdBw+mCWVTD/7Fo5TMhn/buOUJJMV2TglSUlprgONJJlRbDSSZAIvTyNJv0YOWTySSFMZyPJIPuWQN48k2XBmI5Kk2Yp2vYkkabZNXQciSSdenkiS5pm/60AkSSZ22ogkQQ5iX9sSSeIbBt1hi0kyTTqB60AlCYIfcpihjyzzDM11oo8s1jTt9JHJWN6dMdKvkCeMhGXKDjSEkWTdt50kku9KHlwskdaibyyRT3HqTRJpnJGdJJIe/2yniDQp8J0h0mSzN4pIWptqo4iMSkZ+HSgiTXpyp4ik2XFwHSgiEeDZYThaSJoV7+tEC8nGoNgrLS1kmmdyrhMvZJpHAa8DL2SZbHTXiQsSVczsQMP/mJS85zrRP1oH50X/mJcRfbM/5nm2+TqxPyalob5O5I84sFsdaMgfaTZUXCfCR1J6nutE+Gi0wYvw0TjmL8LHp7x7IHw0XvuL8JGWvXgRPj7VzgPh43adY3w0W+zF+Eja1nSdCB+tlnwRPpqU6ovjkVYu70XxSEZrvSgeaUWAL4ZHmpQT14nh0frFL4ZHFD7tBDiGx7y8xxfDYzURxYvjMT015APHY5of5bhOHI9KHHmdGB7TL3aHGVLHNPkarhOto0lvvGgdST+ucZ14HY0teBE7mnTZi9jRaPSd2NGKmyd2tAp9I3Y0Ct3zOhqPe+N1NCnAndfR6vOdyhFF7mpAx+UIv6c70HA5pvm9wuvE32j9xxdn4/0ZjuvE2bg/03E2Iodor3SkjTbD/yJqzCa98SJqrJN54ToRNSLdTQ40VI1pfjT0OtEz0lPOP/Az0s9e59gZSbmYrgM7o1WkGzujScFt7Iw2jtnoGW0cs/Ez2qj1xc+Is9XVgYaf0dnJnaDR6aadlDGZUsuLlBF0On2BlpVR/EYDGVpG6/t5XkZoibVtPTOjHlGw0KJmFN3bDWj4GJ2y3wkZaX6S5ToRMqant+VAwohDPPa2hoRxVQ5ftIvFqsgX7WI1HuqLajGvfNmbaZH9x2qG6tkV09PYtLMr4v3NtFl6RakP35DjV1QuxYUYgkUpOBvIMCw6p8JxLEoSuxrIsCzCgyEDOZbFNLnrrwPNYpoHCq8Tz2KaH8S4jtyKdfKXXQc+Rcntm7FaEkVXo99oFG0Qu/EoSjXZQIZI0VbtNiZFm+7cmRTzPN93HagUi6nnvagUTYlzY1IUx9Zjiz7R79qNQFF6r6oDF2ui1EMM6KkSnd3ZqBIlVDfPtPyIUmDz2MOQaNPOG0WizRJ5jkTnS3qSRJtB9jSJtnC/ESU6z8AzJUoXiIEMVaJtRNm4EgElAxmyREDRQQ9B4n5DS4oIqYwefFgRxUOxmKVFFHH24OJFhKQ40JIh+v28syEmk6F80SEm/46ODZFdG7J3tWyI2M4xO9BSIDp9vFEgynmsakDHgZjm8fLrwIG4T4JnQUwmTtpJEJN+nu46cCCKf10duEgQJYVrMMOCiH3kkIcF0UVsngbxDT08iM657rdjMzSnlcgheuI1FL/zPHsidlF30E2fKMn4aiDDmSieV3fYQ5ToNIAnSpQUETnsYUoUK5UNZukRbeLwRZAIx50cuAgSsWzF3tUwJIo98ZihSLSpvp0kUaxNd+BiRvSCu1Mj2i7GnRpRvAUzIMeHiCkI3YGLEVEExl5paRDFq60OfHgQnZnzRIi2GLORH7ocm2c/ND1Tnv1QumcMYugPnZ709IcukPP8h7iqOujhP3Tq2nMe2obbjfNQGobsOAzRoUSu1WGL3RBiHSxo6A1lp5DDFr+h67p7ExwiBdcdahgOiyla7hSHpnq9MxxCdEN24KI4lGSABS3HoaTTqwMXySF84GDG6lgOJeLKDlw0hxLf2Sstz6Ez5J7nUL4gayDLbShdYeTARW6I8vN9z43dEEdRHfKwG8pHLw1k6A3vrypcb3rD++sh15veENDIDrs5DYWp20KWyFAUmsMWkaFwm3QDWiZDyETwoKUyLKa160VlWH7uSstkKIceqgMXfaE51vHmLHR68kVaaPuN3qSFaZ5Rvg6sheLmmOE62kIJSKsDH95Cc9DkTVYo2nADH4ZCm5jdOApFqWSHPSSFENPuoZul0PY1bjSFMS4vZeMp1M4KCz3chDGuHPfGRwipL91jNwkhsEcnbsyDMa7mqI16UL5M4KGbehBQdNDDPXgzul4v8sEYV3C7kQ8C2q562Afd/vOMg/KxcAMZykFp1iGHPZyDiIzdSy/WQfm+rUMe1sH7i7bXm3VQPguaHXYTD8qXvjz0MA8KK3T32M09eH+t7TpwD8q3tRy2uAeB1e7Bh3BQ9JW/62IcFCmxI7U0g05eN55BPLL4Cxe5oDQ+kwcXu6Dtg3+zC2aTntgYBWVHksMWpaCY8uzBh1NQ3sTfdZEKytfQ7IAsk+AubZZJ0PrtO5Wgdc82LkG5rDtsEQiKcbSvYVkDZTN1By6qQMlAWWixA8qHsjz2MALKvjBv6CgB5QCZhR5KwPvLndeBEvDh4L/epIAxLAdvIwWU7xN56CYClBx1d9jDBCjfZnfQzf4n343xV92Uf/eX4q43zR8ghzw0f/K9Rgc91H5aynHQTed3f0r0elP4yScYq8MeDj/3MM/bJ3f0t3zI+u5vI1xvsr77m9jXm6zv/jje9Sbok68bOegh6Lu/hHq9CfpkGN1jN0Pf/b3P603RN79Vdr1J+e5PlV1vUj65n32WZeWTLVMd+NDyQRbJQzctn3zQ1kEPE598r85ihopPvgFXHfbw7+G64K9bpHuiFLMDF+ueOyviafeE4MFeZ7n2JCqtDjRke8nu+xfZXtlEzJLtSSLLP/Rh27MdeDvDnjhb5MBFsSfpjA18ePVcs9DOrCc5EvtMS633mjvLrQfl7pbSkOvJlwerwxajnmgx/8hFqSeUNfZKy6nn3ZCNSA8i4ibIsucB9NDDnifaxzzRUebJF2/JgYs0L8ZVo91J8+RL0Q5bRHnbMjuiPDla0D34sOPJ8TILPfR49/d8rzdBnpwpdMjNiefMkCfFi2GlPjdaPO2CmZCnwpMlcNBDf2cXwPPfOb/Nc97FYNw2z3N3f4D3ehPdib3LBjPsdvIh7uqwm9Jufkj3erPYCWFcddhNYydfKrPPMjx28snP6rCHvE4+jGoxx14nJNEeXOx1cTK3XCf2OgjtBj7sdffnw64DfZ18RNC+h+Wsc7K8cdY5B2LjqROhrA5cRHX3R4ivA1OdfCOzOnDR093fwrwO9HT3h+GvAz+dfEPRzI2jqMPH/qrDFkWdkAn6Cx+OuuEE3lHUDePUeIq6YRo0N1q6YXxNz0TXTdDsmej6yght5HPsMMfuoId9rps610Y/140T5Ojnul0VT0DXjcPuGei6tUyeda6v1OjOOrfixp11bkNu1rluotCddG64UVuiOWOJPbkcKFXJQYZQLvuLFodcN6d1dg65YfJ9bxK5mrLDFoncsBHvi0QOR1DIo45Ezl+5KOSGOaVxoo1b5+V2qjjshkAOtFxxI2aHLbK4YfKPG1fcsFrLc8Vh1w4HLa64MT8/cx244obJTe9cccPu540tbthwdmOLQ8OBv26xxQ1TwN/p4obJhR8o4rqdl40jbrXa7SRx42HzeLHEjfn1zOtNEzd+dtkNTdxYVY+NJ26YLKinhhs/KwiWGq7/7CRaarhuIlhHDdeNw+Gp4bpJ/XpqOKsTPRtc/62d7djgusnKODq4buyYZ4Drpslpo4Drq5llY31DN4W9yDC99flJkevA9Dbmh8CvA9MbfLKlXjauN/kApH2gJXvzqb2N7U38r+7Bh+5NDKq/cnG8iYNjn2lJ3iSwrB68ad7EC+sOWzxvsi88uMjd5qe6rxO3mwusPLebhJ0OWtxu8nlUf90id8NgQnfgYncb/qaW3A088B56yN2GSU1s3G73t8GvA7nbMB3yO7kbVsnOp2N0G/Mrp9ebxW3MjwRcbxo3nA63kOVxGzY57rnbhl9YS95mVY8nbxurNrnxtVmt5Pnahgm9PF8bPljmoZuj7dFk/x9EGVE50qEAAA=="
obs = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(PAYLOAD))))
obs['Date'] = pd.to_datetime(obs['Date'])
obs['year'] = obs.Date.dt.year
obs['era'] = np.where(obs.year <= 2011, 'pre', 'mining')
m = obs[obs.station == 'WLd'].copy().rename(columns={'WL': 'WL263'})
q2 = obs[obs.station == 'WLu'].copy().rename(columns={'WL': 'WL262'})


def fit(d, col):
    s, i, r, p, se = stats.linregress(np.log10(d.Q), d[col])
    return s, i, r


s263p, i263p, _ = fit(m[m.era == 'pre'], 'WL263')
s263m, i263m, _ = fit(m[m.era == 'mining'], 'WL263')
s262p, i262p, _ = fit(q2[q2.era == 'pre'], 'WL262')
s262m, i262m, _ = fit(q2[q2.era == 'mining'], 'WL262')
annq = q2.groupby('year').Q.mean()
sl, ic, _, _, _ = stats.linregress(annq.index, annq.values)
mk_pval = mk_p(annq.values)
m['resid'] = m.WL263 - (i263p + s263p * np.log10(m.Q))
q2['resid'] = q2.WL262 - (i262p + s262p * np.log10(q2.Q))
res_WLd = m.groupby('year').resid.mean()
res_WLu = q2.groupby('year').resid.mean()
print(f'WLd pre a={i263p:.3f} b={s263p:.3f} | mining a={i263m:.3f} b={s263m:.3f}')
print(f'WLu pre a={i262p:.3f} b={s262p:.3f} | mining a={i262m:.3f} b={s262m:.3f}')
print(f'Annual Q OLS {sl:+.2f} m3/s/yr, MK p={mk_pval:.3f}')

C_WLd, C_WLu, CPRE = '#C0392B', '#2471A3', '#7f8c8d'
WB = dict(facecolor='white', edgecolor='none', alpha=0.0, pad=2.0)

fig, ax = plt.subplots(2, 2, figsize=(FIG_W, FIG_H),
                       gridspec_kw={'hspace': 0.26, 'wspace': 0.24})
(a, b), (c, d) = ax
chk = LayoutCheck(fig)
chk.ink_threshold = 200


def plabel(ax_, txt):
    return chk.label(f'{txt}:letter',
                     ax_.text(0.985, 0.965, txt, transform=ax_.transAxes,
                              fontsize=FS['panel'], fontweight='bold',
                              va='top', ha='right'), ax_)


# ------------------------------------------------------------------ (a)
a.bar(annq.index, annq.values, color='#5DADE2', alpha=.75, width=.8, label='Annual mean Q')
roll = annq.rolling(5, center=True, min_periods=3).mean()
a.plot(roll.index, roll.values, 'k-', lw=2, label='5-yr rolling mean')
xs = np.array([annq.index.min(), annq.index.max()])
a.plot(xs, ic + sl * xs, '--', color='#555', lw=1.6,
       label=f'Trend {sl:+.1f} m\u00b3/s/yr\n(MK p={mk_pval:.2f}, n.s.)')
a.set_xlabel('Year')
chk.label('a:ylabel', a.set_ylabel('Mean annual discharge (m\u00b3/s)'), a, inside=False)
a.set_ylim(0, 600)
lega = a.legend(loc='upper right', fontsize=FS['legend'], borderpad=0.25,
                labelspacing=0.15, handlelength=HL, borderaxespad=0.55, framealpha=0.95)
chk.legend(lega, 'a:legend')
a.grid(alpha=.3)
chk.label('(a):letter', a.text(0.018, 0.965, '(a)', transform=a.transAxes,
                              fontsize=FS['panel'], fontweight='bold',
                              va='top', ha='left'), a)
chk.bars(a, annq.index.values, annq.values, width=0.8)
chk.path(a, roll.dropna().index.values.astype(float), roll.dropna().values)
chk.seg(a, xs[0], xs[1], ic + sl * xs[0], ic + sl * xs[1])

# ------------------------------------------------------------------ (b)
mp, mm2 = m[m.era == 'pre'], m[m.era == 'mining']
qp, qm = q2[q2.era == 'pre'], q2[q2.era == 'mining']
DS = 5
b.scatter(mp.Q, mp.WL263, s=DS, c=CPRE, alpha=.20, marker='o', linewidths=0)
b.scatter(qp.Q, qp.WL262, s=DS, c=CPRE, alpha=.20, marker='o', linewidths=0)
b.scatter(mm2.Q, mm2.WL263, s=DS + 2, c=C_WLd, alpha=.22, marker='^', linewidths=0)
b.scatter(qm.Q, qm.WL262, s=DS + 2, c=C_WLu, alpha=.22, marker='s', linewidths=0)
xr = np.logspace(np.log10(5), np.log10(2902), 100); lx = np.log10(xr)
b.plot(xr, i263p + s263p * lx, color=C_WLd, lw=2.5, ls=(0, (6, 2)),
       label='WLd pre-mining (≤2011)')
b.plot(xr, i263m + s263m * lx, color=C_WLd, lw=2.5, ls='-', label='WLd mining (≥2012)')
b.plot(xr, i262p + s262p * lx, color=C_WLu, lw=2.5, ls=(0, (6, 2)),
       label='WLu pre-mining (≤2011)')
b.plot(xr, i262m + s262m * lx, color=C_WLu, lw=2.5, ls='-', label='WLu mining (≥2012)')
b.set_xscale('log'); b.set_ylim(5.4, 24.5)
b.set_xlabel('Discharge at WLu (m\u00b3/s, log scale)')
chk.label('b:ylabel', b.set_ylabel('Water level (m PWD)'), b, inside=False)
legb = b.legend(loc='upper left', ncol=1, fontsize=FS['legend'], framealpha=.95,
                borderpad=0.4, labelspacing=0.3, handlelength=HL,
                columnspacing=1.0, borderaxespad=0.7)                       # 262
chk.legend(legb, 'b:legend')
b.grid(alpha=.25, which='both'); plabel(b, '(b)')
for dd, col in [(mp, 'WL263'), (mm2, 'WL263')]:
    chk.pts(b, dd.Q.values, dd[col].values)
for dd in (qp, qm):
    chk.pts(b, dd.Q.values, dd.WL262.values)
for i0, s0 in [(i263p, s263p), (i263m, s263m), (i262p, s262p), (i262m, s262m)]:
    chk.path(b, xr, i0 + s0 * lx, n=3)

# ------------------------------------------------------------------ (c)
Qs = [20, 50, 100, 200, 500, 1000]; lq = np.log10(Qs)
d263 = (i263p + s263p * lq) - (i263m + s263m * lq)
d262 = (i262p + s262p * lq) - (i262m + s262m * lq)
xpos = np.arange(len(Qs)); wd = .38
c.bar(xpos - wd / 2, d263, wd, color=C_WLd, label='WLd (downstream)')
c.bar(xpos + wd / 2, d262, wd, color=C_WLu, label='WLu (upstream)')
for k, (xi, v) in enumerate(zip(xpos - wd / 2, d263)):
    chk.label(f'c:vd{k}', c.text(xi, v + .10, f'{v:.1f}', ha='center', va='bottom', rotation=90,
                                 fontsize=FS['annot'], color=C_WLd, fontweight='bold'), c)
for k, (xi, v) in enumerate(zip(xpos + wd / 2, d262)):
    chk.label(f'c:vu{k}', c.text(xi, v + .45, f'{v:.1f}', ha='center', va='bottom', rotation=90,
                                 fontsize=FS['annot'], color=C_WLu, fontweight='bold'), c)
c.set_xticks(xpos); c.set_xticklabels(['20', '50', '100', '200', '500', '1,000'])
c.set_xlabel('Discharge, Q (m³/s)')
chk.label('c:ylabel', c.set_ylabel('Water level drop (m)'), c, inside=False)
c.set_ylim(0, 3.95)
legc = c.legend(loc='upper left', fontsize=FS['legend'], borderpad=0.4,
                labelspacing=0.3, handlelength=HL, borderaxespad=0.7, framealpha=0.95)
chk.legend(legc, 'c:legend')
c.grid(alpha=.3, axis='y'); plabel(c, '(c)')
chk.bars(c, xpos - wd / 2, d263, width=wd)
chk.bars(c, xpos + wd / 2, d262, width=wd)

# ------------------------------------------------------------------ (d)
d.set_ylim(-4.05, 1.95)
d.axhline(0, color='k', lw=.9)
d.bar(res_WLd.index, res_WLd.values, color=C_WLd, alpha=.40, width=.42, align='edge')
r3 = res_WLd.rolling(3, center=True, min_periods=2).mean()
r6 = res_WLu.rolling(3, center=True, min_periods=2).mean()
d.plot(r3.index, r3.values, color=C_WLd, lw=2.3, label='WLd (3-yr mean)')
d.plot(r6.index, r6.values, color=C_WLu, lw=2.3, label='WLu (3-yr mean)')
d.vlines(2011.5, -2.15, 1.95, color='#555', ls=':', lw=1.4)
chk.label('d:2012', d.text(2012.3, 1.80, '2012', color='#555', fontsize=FS['annot'],
                           va='top', ha='left'), d)
chk.label('d:lead', d.text(0.015, 0.975, 'WLd leads (~2011)', transform=d.transAxes,
                           color=C_WLd, fontsize=FS['annot'], style='italic',
                           va='top', ha='left', bbox=WB), d)
chk.label('d:follow', d.text(0.015, 0.845, 'WLu follows (~2017)',
                             transform=d.transAxes, color=C_WLu, fontsize=FS['annot'],
                             style='italic', va='top', ha='left', bbox=WB), d)
d.set_xlabel('Year')
chk.label('d:ylabel', d.set_ylabel('Stage anomaly (m)'), d, inside=False)
legd = d.legend(loc='lower left', fontsize=FS['legend'], borderpad=0.4,
                labelspacing=0.3, handlelength=HL, borderaxespad=0.7, framealpha=0.95)
chk.legend(legd, 'd:legend')
d.grid(alpha=.3); plabel(d, '(d)')
chk.bars(d, res_WLd.index.values + 0.21, res_WLd.values, width=0.42)
chk.path(d, r3.index.values.astype(float), r3.values)
chk.path(d, r6.index.values.astype(float), r6.values)
chk.seg(d, res_WLd.index.min() - 1, res_WLd.index.max() + 1, 0, 0)

fig.align_ylabels([a, c]); fig.align_ylabels([b, d])

issues = chk.run()
if issues:
    print(f'\n{len(issues)} layout issue(s) above — figure not saved.')
else:
    fig.savefig('fig10_discharge_v4.png', dpi=300, bbox_inches='tight')
    fig.savefig('fig10_discharge_v4.pdf', bbox_inches='tight')
    print('saved fig10_discharge_v4.png and fig10_discharge_v4.pdf')
plt.show()
plt.close(fig)


## Figure 11: paired-well groundwater attribution

Both post-break trends fitted over 2013 to 2024, GWn at +1.72 m/decade and
GWref at -0.47 m/decade. The y-axis now says annual mean, which is what
separates these values from the seasonal ones in Figure 7.


In [ ]:
# @title Figure 11 - paired-well groundwater attribution
# ---------------------------------------------------------------- chat 31
# FIG_SCALE rescales the drawing canvas while the font sizes below are left as
# authored. AGU prints a two-column figure at 170 mm (6.693 in), so text set at
# F pt on a canvas W inches wide reaches the page at F * 6.693 / W pt. Scaling
# the canvas down is therefore the same as enlarging the type on the printed
# page, and it preserves every layout decision in relative terms.
FIG_SCALE_W, FIG_SCALE_H = 1.192, 1.192
# ---------------------------------------------------------------------------
# All packages (numpy, pandas, scipy, matplotlib) are pre-installed in Colab.
# Nothing to install — just run the cells in order.



# Data embedded directly — no file upload needed
# Source: BWDB wells GT7218003 (GWn, within mined reach) and GT7218004 (GWref, outside mined reach)
# Annual mean water-table depth (m bgl). GWn missing 2013 and 1997 (measurement gaps).
# To update the data, edit the list below.

import pandas as pd

_DATA = [{"year": 1988, "GWn": 2.657, "GWref": 3.399}, {"year": 1989, "GWn": 3.406, "GWref": 3.718}, {"year": 1990, "GWn": 3.139, "GWref": 3.317}, {"year": 1991, "GWn": 2.75, "GWref": 3.296}, {"year": 1992, "GWn": 2.798, "GWref": 3.314}, {"year": 1993, "GWn": 2.692, "GWref": 3.328}, {"year": 1994, "GWn": 2.829, "GWref": 3.342}, {"year": 1995, "GWn": 2.897, "GWref": 3.605}, {"year": 1996, "GWn": 2.817, "GWref": 3.66}, {"year": 1998, "GWn": 2.638, "GWref": 3.626}, {"year": 1999, "GWn": 3.098, "GWref": 3.985}, {"year": 2000, "GWn": 2.605, "GWref": 3.583}, {"year": 2001, "GWn": 2.911, "GWref": 3.995}, {"year": 2002, "GWn": 2.597, "GWref": 3.626}, {"year": 2003, "GWn": 2.869, "GWref": 3.674}, {"year": 2004, "GWn": 2.716, "GWref": 3.985}, {"year": 2005, "GWn": 2.821, "GWref": 4.081}, {"year": 2006, "GWn": 3.014, "GWref": 4.418}, {"year": 2007, "GWn": 2.976, "GWref": 4.091}, {"year": 2008, "GWn": 2.903, "GWref": 4.306}, {"year": 2009, "GWn": 2.959, "GWref": 4.915}, {"year": 2010, "GWn": 3.576, "GWref": 4.751}, {"year": 2011, "GWn": 3.421, "GWref": 5.178}, {"year": 2012, "GWn": 3.23, "GWref": 5.277}, {"year": 2013, "GWn": None, "GWref": 5.173}, {"year": 2014, "GWn": 3.215, "GWref": 4.984}, {"year": 2015, "GWn": 3.399, "GWref": 4.34}, {"year": 2016, "GWn": 3.626, "GWref": 4.933}, {"year": 2017, "GWn": 3.349, "GWref": 4.358}, {"year": 2018, "GWn": 4.045, "GWref": 4.472}, {"year": 2019, "GWn": 4.112, "GWref": 4.419}, {"year": 2020, "GWn": 4.203, "GWref": 4.547}, {"year": 2021, "GWn": 4.285, "GWref": 4.468}, {"year": 2022, "GWn": 4.542, "GWref": 4.521}, {"year": 2023, "GWn": 4.848, "GWref": 4.392}, {"year": 2024, "GWn": 4.894, "GWref": 4.483}]

d = pd.DataFrame(_DATA)   # columns: year, GWn, GWref
print(f"{len(d)} years loaded ({int(d.year.min())}–{int(d.year.max())}); GWn missing: {d[d.GWn.isna()].year.tolist()}")
d.head(10)


import numpy as np
from scipy import stats

# Pre-mining fit (1988–2011, n = 23 complete pairs)
pre = d[(d.year <= 2011) & d.GWn.notna() & d.GWref.notna()]
sl, ic, r, p, _ = stats.linregress(pre.GWref, pre.GWn)

res  = pre.GWn - (ic + sl * pre.GWref)
n    = len(pre)
syx  = np.sqrt((res ** 2).sum() / (n - 2))
tc   = stats.t.ppf(0.975, n - 2)
mx   = pre.GWref.mean()
sxx  = ((pre.GWref - mx) ** 2).sum()

d["pred"]   = ic + sl * d.GWref
d["excess"] = d.GWn - d.pred
d["pi"]     = tc * syx * np.sqrt(1 + 1/n + (d.GWref - mx)**2 / sxx)

def trend(sub, col):
    s = sub.dropna(subset=[col])
    m, b, rr, pp, _ = stats.linregress(s.year, s[col])
    return m, b, pp, s.year.min(), s.year.max()

m_pre, b_pre, p_pre, y0_pre, y1_pre = trend(d[d.year <= 2011], "GWn")
m_min, b_min, p_min, y0_min, y1_min = trend(d[d.year >= 2013], "GWn")
m_ref, b_ref, p_ref, y0_ref, y1_ref = trend(d[d.year >= 2013], "GWref")
exc_mech = d[d.year >= 2017].excess.mean()

print(f"Pre-mining fit:  GWn = {ic:.3f} + {sl:.3f}·GWref   r = {r:.2f}  p = {p:.4f}  n = {n}")
print(f"GWn trend  pre-mining : {m_pre*10:+.2f} m/decade  (p = {p_pre:.2f})")
print(f"GWn trend  2013–2024  : {m_min*10:+.2f} m/decade  (p = {p_min:.4f})")
print(f"GWref trend 2013–2024 : {m_ref*10:+.2f} m/decade  (p = {p_ref:.4f})")
print(f"Mean excess, mechanized (2017–2024): {exc_mech:+.3f} m")



import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ── Style ─────────────────────────────────────────────────────────────────────
VERM, BLUE, GREY, INK = '#D55E00', '#0072B2', '#7A7A7A', '#1E2A30'
ERA_FILL = {'pre': '#FFFFFF', 'early': '#F3EBE4', 'mech': '#EBDCD0'}

# Era boundaries follow the manuscript:
#   Pre-mining 1988–2011 | Early mining 2012–2016 | Mechanized mining 2017–present
B1, B2 = 2011.5, 2016.5
X0, X1 = 1987.2, 2024.9

import matplotlib as _mpl
_mpl.rcParams.update(_RC_DEFAULTS)   # clear any style left by an earlier cell
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 12,
    'axes.labelsize': 12.5,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10.4,
    'axes.linewidth': 0.9,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
})

fig, (axa, axb) = plt.subplots(
    2, 1, figsize=(8.2 * FIG_SCALE_W, 7.5 * FIG_SCALE_H), dpi=150, sharex=True,
    gridspec_kw={'height_ratios': [1.35, 1.0], 'hspace': 0.0}
)  # use dpi=300 for the final saved PNG
fig.patch.set_facecolor('white')

def eras(ax):
    ax.axvspan(X0, B1, color=ERA_FILL['pre'],   zorder=0)
    ax.axvspan(B1, B2, color=ERA_FILL['early'],  zorder=0)
    ax.axvspan(B2, X1, color=ERA_FILL['mech'],   zorder=0)
    for b in (B1, B2):
        ax.axvline(b, color='#B0A79F', lw=0.8, zorder=1)

# ── Panel (a): water-table depth ──────────────────────────────────────────────
eras(axa)
axa.plot(d.year, d.GWn,   '-o', color=VERM, ms=3.6, lw=1.5, zorder=5)
axa.plot(d.year, d.GWref, '-o', color=BLUE, ms=3.6, lw=1.5, zorder=5)

xs = np.array([y0_pre, y1_pre])
axa.plot(xs, b_pre + m_pre * xs, ls='--', lw=1.5, color=GREY,      zorder=6)
xs = np.array([y0_min, y1_min])
axa.plot(xs, b_min + m_min * xs, ls='--', lw=1.7, color='#111111', zorder=6)
xs = np.array([y0_ref, y1_ref])
axa.plot(xs, b_ref + m_ref * xs, ls='--', lw=1.5, color=BLUE, alpha=0.85, zorder=6)

axa.set_ylim(5.95, 2.15)          # inverted axis: deeper = downward
axa.set_ylabel('Annual mean water-table depth (m bgl)', color=INK)
axa.tick_params(axis='x', which='both', length=0, labelbottom=False)
axa.grid(axis='y', color='#D9D9D9', lw=0.6, zorder=0)
axa.set_axisbelow(True)

hnd = [
    Line2D([], [], color=VERM, marker='o', ms=4, lw=1.5, label='GWn'),
    Line2D([], [], color=BLUE, marker='o', ms=4, lw=1.5, label='GWref'),
    Line2D([], [], color=GREY,      ls='--', lw=1.5,
           label=f'GWn, pre-mining: {m_pre*10:+.2f} m/decade (n.s.)'),
    Line2D([], [], color='#111111', ls='--', lw=1.7,
           label=f'GWn, 2013–2024: {m_min*10:+.2f} m/decade'),
    Line2D([], [], color=BLUE,      ls='--', lw=1.5,
           label=f'GWref, 2013–2024: {m_ref*10:+.2f} m/decade'.replace('-', '\u2212')),
]
axa.legend(handles=hnd, loc='lower left', fontsize=10.4, frameon=True,
           framealpha=0.95, edgecolor='#CCCCCC', borderpad=0.5,
           labelspacing=0.40, handlelength=2.4, borderaxespad=1.0).set_zorder(10)
axa.text(0.986, 0.955, '(a)', transform=axa.transAxes, ha='right', va='top',
         fontsize=14, fontweight='bold', color=INK, zorder=11)

# ── Panel (b): excess deepening ───────────────────────────────────────────────
eras(axb)
axb.fill_between(d.year, -d.pi, d.pi, color='#BFBFBF', alpha=0.45, lw=0, zorder=2)
axb.axhline(0, color='#5A5A5A', lw=0.9, ls=':', zorder=3)
axb.plot([2017, 2024], [exc_mech]*2, ls='--', lw=1.7, color='#111111', zorder=6)
axb.plot(d.year, d.excess, '-o', color=VERM, ms=3.8, lw=1.6, zorder=7)

axb.set_ylim(-1.35, 2.45)
axb.set_xlim(X0, X1)
axb.set_ylabel('Excess deepening (m)', color=INK)
axb.set_xlabel('Year', color=INK)
axb.set_xticks(np.arange(1990, 2025, 5))
axb.grid(axis='y', color='#D9D9D9', lw=0.6, zorder=0)
axb.set_axisbelow(True)

hnd_b = [
    Line2D([], [], color=VERM, marker='o', ms=4, lw=1.6, label='Excess deepening'),
    Patch(facecolor='#BFBFBF', alpha=0.45,                label='95% prediction band'),
    Line2D([], [], color='#111111', ls='--', lw=1.7,
           label=f'Mean, mechanized: {exc_mech:+.2f} m'),
]
axb.legend(handles=hnd_b, loc='upper left', fontsize=10.4, frameon=True,
           framealpha=0.95, edgecolor='#CCCCCC', borderpad=0.5,
           labelspacing=0.40, handlelength=2.4, borderaxespad=1.0).set_zorder(10)
axb.text(0.986, 0.955, '(b)', transform=axb.transAxes, ha='right', va='top',
         fontsize=14, fontweight='bold', color=INK, zorder=11)

# Era names in empty strip at base of panel (b)
ERA_Y = -1.02
for xc, name in [
    ((X0 + B1) / 2, 'Pre-mining'),
    ((B1 + B2) / 2, 'Early\nmining'),
    ((B2 + X1) / 2, 'Mechanized\nmining'),
]:
    axb.text(xc, ERA_Y, name, ha='center', va='center', fontsize=10.4,
             color='#5A4B3F', fontweight='bold', linespacing=1.15, zorder=8)

plt.show()
print('Figure displayed.')


# ── Save at 300 dpi and download ─────────────────────────────────────────────
OUT = 'fig11_groundwater_reference.png'
fig.savefig(OUT, dpi=300, facecolor='white', bbox_inches='tight')
fig.savefig('fig11_groundwater_reference.pdf', facecolor='white', bbox_inches='tight')
plt.close(fig)
print(f'Saved: {OUT}')

# ── Summary table of key statistics ─────────────────────────────────────────
import pandas as pd
rows = [
    ('Pre-mining fit', f'GWn = {ic:.3f} + {sl:.3f}·GWref', f'r = {r:.2f}, p = {p:.4f}, n = {n}'),
    ('GWn trend, pre-mining', f'{m_pre*10:+.2f} m/decade', f'p = {p_pre:.2f} (n.s.)'),
    ('GWn trend, 2013–2024', f'{m_min*10:+.2f} m/decade', f'p = {p_min:.4f}'),
    ('GWref trend, 2013–2024', f'{m_ref*10:+.2f} m/decade', f'p = {p_ref:.4f}'),
    ('Mean excess, mechanized mining (2017–2024)', f'{exc_mech:+.3f} m', ''),
]
pd.DataFrame(rows, columns=['Statistic', 'Value', 'Notes'])


## Verification

Run after all seven figures. Measures what a reader sees on the page, not
what was passed to matplotlib.


In [ ]:
# @title Validate print-size readability
import glob, os
import numpy as np
from PIL import Image
from scipy import ndimage

"""Printed-size audit.

Each figure is rescaled to AGU's 170 mm two-column width and the rendered
glyphs are measured, so the number reported is the size a reader sees on the
page rather than the size passed to matplotlib.
"""

PAGE_MM = 170.0
PAGE_IN = PAGE_MM / 25.4
FLOOR_PT = 8.0                    # AGU minimum for figure labelling

# build-time tick-label size declared in each cell
DECLARED = {
    'fig4_barArea_v4.png': 14, 'fig5_cross_sections_final.png': 9.5,
    'fig6_water_level_final.png': 9.5, 'cl_fig_gwt_v3.png': 14,
    'someshwari_fig9_rainfall_v10.png': 14, 'fig10_discharge_v4.png': 14,
    'fig11_groundwater_reference.png': 11,
}


def printed(path, declared):
    im = Image.open(path).convert('L')
    w, h = im.size
    t = int(PAGE_IN * 300)
    a = np.array(im.resize((t, int(h * t / w)), Image.LANCZOS))
    lab, _ = ndimage.label(a < 128)
    hs = [s[0].stop - s[0].start for s in ndimage.find_objects(lab)
          if 12 < s[0].stop - s[0].start < 45 and 3 < s[1].stop - s[1].start < 45
          and (s[0].stop - s[0].start) >= (s[1].stop - s[1].start) * 0.8]
    cap = np.percentile(hs, 75) / 300 * 72 / 0.729     # digits and capitals
    return declared * PAGE_IN / (w / 300), cap, h / w * PAGE_MM


print(f"{'figure':34s}{'tick pt':>9s}{'cap pt':>8s}{'height mm':>11s}")
worst = 99.0
for name, dec in DECLARED.items():
    if not os.path.exists(name):
        print(f'{name:34s}   not built')
        continue
    tick, cap, hmm = printed(name, dec)
    worst = min(worst, tick)
    note = '' if tick >= FLOOR_PT else '   BELOW THE 8 pt FLOOR'
    tall = '' if hmm <= 228 else '   OVER THE 228 mm HEIGHT CAP'
    print(f'{name:34s}{tick:9.2f}{cap:8.1f}{hmm:11.0f}{note}{tall}')

print()
print(f'smallest printed tick size across the set: {worst:.2f} pt '
      f'({"pass" if worst >= FLOOR_PT else "FAIL"})')
